In [1]:
import pandas as pd
import xml.etree.ElementTree as ET
from sklearn.preprocessing import MinMaxScaler
from tqdm import tqdm

## Data

### PPI

In [2]:
protein_interaction = pd.read_csv('Data/Protein-protein interaction data/9606.protein.links.v12.0.txt', sep= ' ')
protein_interaction_full = pd.read_csv('Data/Protein-protein interaction data/9606.protein.links.full.v12.0.txt', sep= ' ')
protein_interaction_detailed = pd.read_csv('Data/Protein-protein interaction data/9606.protein.links.detailed.v12.0.txt', sep= ' ')
### convert proteins to their true names
protein_info = pd.read_csv('Data/Protein-protein interaction data/9606.protein.info.v12.0.txt', on_bad_lines='skip', sep='\t')
protein_aliases= pd.read_csv('Data/Protein-protein interaction data/9606.protein.aliases.v12.0.txt', on_bad_lines='skip', sep='\t')

In [3]:
### convert proteins to their true names
protein_info = pd.read_csv('Data/Protein-protein interaction data/9606.protein.info.v12.0.txt', on_bad_lines='skip', sep='\t')
protein_aliases= pd.read_csv('Data/Protein-protein interaction data/9606.protein.aliases.v12.0.txt', on_bad_lines='skip', sep='\t')

# Method 1: Using the to_dict() method with 'index' as orient
protein_info_translate_name_dict = protein_info.set_index('#string_protein_id')['preferred_name'].to_dict()
protein_alias_translate_name_dict = protein_aliases.set_index('#string_protein_id')['alias'].to_dict()
#print(protein_info_translate_name_dict)

### Protein1
protein1_name = []
for prot_id in tqdm(protein_interaction['protein1']):
    if prot_id in protein_info_translate_name_dict:
        protein1_name.append(protein_info_translate_name_dict[prot_id])
    elif prot_id in protein_alias_translate_name_dict:
        protein1_name.append(protein_alias_translate_name_dict[prot_id])
    else:
        protein1_name.append('')

### Protein 2
protein2_name = []
for prot_id in tqdm(protein_interaction['protein2']):
    if prot_id in protein_info_translate_name_dict:
        protein2_name.append(protein_info_translate_name_dict[prot_id])
    elif prot_id in protein_alias_translate_name_dict:
        protein2_name.append(protein_alias_translate_name_dict[prot_id])
    else:
        protein2_name.append('')

protein_interaction['Translated_protein_1'] = protein1_name
protein_interaction['Translated_protein_2'] = protein2_name

# Create a set of all (protein1, protein2) pairs
ppi_pairs = set(zip(protein_interaction['Translated_protein_1'], protein_interaction['Translated_protein_2']))
# Check for missing reverse pairs
missing_reverse = []
for a, b in ppi_pairs:
    if (b, a) not in ppi_pairs:
        missing_reverse.append((a, b))

print(f"Number of pairs missing their reverse: {len(missing_reverse)}")
if missing_reverse:
    print("Examples:", missing_reverse[:10])
else:
    print("All pairs have their reverse present.")

100%|██████████| 13715404/13715404 [00:02<00:00, 4683321.97it/s]


Number of pairs missing their reverse: 0
All pairs have their reverse present.


In [4]:
protein_interaction

,protein1,protein2,combined_score,Translated_protein_1,Translated_protein_2
0,9606.ENSP00000000233,9606.ENSP00000356607,173,ARF5,RALGPS2
1,9606.ENSP00000000233,9606.ENSP00000427567,154,ARF5,FHDC1
2,9606.ENSP00000000233,9606.ENSP00000253413,151,ARF5,ATP6V1E1
3,9606.ENSP00000000233,9606.ENSP00000493357,471,ARF5,CYTH2
4,9606.ENSP00000000233,9606.ENSP00000324127,201,ARF5,PSD3
...,...,...,...,...,...
13715399,9606.ENSP00000501317,9606.ENSP00000475489,195,RFX7,MPHOSPH9
13715400,9606.ENSP00000501317,9606.ENSP00000370447,158,RFX7,VCX
13715401,9606.ENSP00000501317,9606.ENSP00000312272,226,RFX7,YPEL2
13715402,9606.ENSP00000501317,9606.ENSP00000402092,169,RFX7,SAMD3


### DrugBank

In [5]:
# import xml.etree.ElementTree as ET

# # Load XML
# drugbank_xml = 'Data/DGIDB/drug_bank.xml'
# tree = ET.parse(drugbank_xml)
# root = tree.getroot()

# # Namespace
# ns = {'db': 'http://www.drugbank.ca'}

# Helper to clean tag names
def clean_tag(tag):
    return tag.split('}')[-1] if '}' in tag else tag

# Recursive function to print structure
def print_structure(elem, level=0):
    indent = '  ' * level
    print(f"{indent}- {clean_tag(elem.tag)}")
    for child in elem:
        print_structure(child, level + 1)

# # Get first drug
# first_drug = root.find('db:drug', ns)

# print("🌿 Structure of First Drug Entry:")
# print_structure(first_drug)
# print("\n🌳 Structure of First 3 Drug Entries:")
# drugs = root.findall('db:drug', ns)

# for i, drug in enumerate(drugs[:3]):
#     print(f"\n🔬 Drug {i+1}:")
#     print_structure(drug)


In [6]:
def structure_drug_bank_data(drug_bank_file = 'Data/DGIDB/drug_bank.xml'):
    """
    Function to structure the drug bank data from the XML file.
    :param drug_bank_file: Path to the drug bank XML file.
    :return: DataFrame containing structured drug bank data.
    """
    ### FYI the .find command only finds the first instance of a tag, 
    ### while .findall retrieves all instances of the specified tag within the current element.

    tree = ET.parse(drug_bank_file)
    root = tree.getroot()

    # DrugBank uses a specific namespace
    ns = {'db': 'http://www.drugbank.ca'}
    ### extract all drug elements
    drugs = root.findall('db:drug', ns)
    print(f"Found {len(drugs)} drugs in the DrugBank XML.")
    # Extract drug-gene interactions
    interactions = []
    # The interactions list will store dictionaries with 'drug' and 'gene' keys.
    for drug in root.findall('db:drug', ns): # root.findall('db:drug', ns): Finds all <drug> elements using the namespace.
        drug_name  = drug.find('db:name', ns).text  # drug.find('db:name', ns): Gets the drug's name.
        # print(drug_name)
        for target in drug.findall('db:targets/db:target', ns):  # drug.findall('db:targets/db:target', ns): Finds all <target> elements within <targets>.
            # print(target.tag)
            gene_description = target.find('db:name', ns)  # target.find('db:name', ns): Extracts the gene name for each target.
            poly = target.find('db:polypeptide', ns)  # target.find('db:polypeptide', ns): Extracts the polypeptide information.
            action = target.find('db:actions/db:action', ns) # target.find('db:actions/db:action', ns): Extracts the action of the drug on the target.
            if poly is not None:
                poly_name = poly.find('db:name', ns)
                gene_name = poly.find('db:gene-name', ns)
                specific_function = poly.find('db:specific-function', ns)
                interactions.append({
                    'drug': drug_name,
                    'polypeptide': poly_name.text if poly_name is not None else None,
                    'gene': gene_name.text if gene_name is not None else None,
                    'gene_description': gene_description.text if gene_description is not None else None,
                    'action': action.text if action is not None else None,
                    'specific_function': specific_function.text if specific_function is not None else None
                })
            ############# if polypeptide is not present, we still want to add the drug and gene information
            ############# this is because some drugs may not have a polypeptide associated with them
            ############# but we still want to capture the drug and gene information
            ############# this is common in the DrugBank database, where some drugs target genes directly
            ############# and do not have a polypeptide associated with them

            else:
                gene_name = None
                specific_function = None
                poly_name = None
                action = None
                gene_description = None
                resource = None
                identifier = None
  
                interactions.append({
                        'drug': drug_name,
                        'polypeptide': poly_name.text if poly_name is not None else None,
                        'gene': gene_name.text if gene_name is not None else None,
                        'gene_description': gene_description.text if gene_description is not None else None,
                        'action': action.text if action is not None else None,
                        'specific_function': specific_function.text if specific_function is not None else None
                    })
        
    # Convert to DataFrame
    # Converts the list of dictionaries into a pandas DataFrame, which is easier to analyze, filter, and export.
    df = pd.DataFrame(interactions)

    return df

In [7]:
Drug_bank = structure_drug_bank_data('Data/DGIDB/drug_bank.xml')

Found 17430 drugs in the DrugBank XML.


In [8]:
Drug_bank['drug'].unique()
print(f"Total unique drugs: {len(Drug_bank['gene'].unique())}")

Total unique drugs: 4187


In [9]:
Drug_bank_temp = Drug_bank[~Drug_bank['gene'].isnull()]
Drug_bank_temp['gene'].nunique()

4186

### Genetic results

In [10]:
### import data

### genes
hpv_positive_genes  = pd.read_csv('Results/CNV results/HPV positive CNV top genes.csv')
hpv_negative_genes = pd.read_csv('Results/CNV results/HPV negative CNV top genes.csv')

### drug candiates
hpv_positive_direct_drug_candidates = pd.read_csv('Results/CNV results/HPV Positive Top Direct Drug Candidates Aggregated.csv')
hpv_positive_indirect_drug_candidates = pd.read_csv('Results/CNV results/HPV Positive Top Indirect Drug Candidates Aggregated.csv')

hpv_negative_direct_drug_candidates = pd.read_csv('Results/CNV results/HPV Negative Top Direct Drug Candidates Aggregated.csv')
#### no direct drug candidates came from Deletions, only amplifications
hpv_negative_direct_drug_candidates['MUT_TYPE'] = 'AMPLIFICATION'
hpv_negative_indirect_drug_candidates = pd.read_csv('Results/CNV results/HPV Negative Top Indirect Drug Candidates Aggregated.csv')

### somatic mtuation
hpv_positive_som_genes = pd.read_csv('Results/SOM results/HPV positive top genes.csv')
hpv_positive_som_direct_drug_candidates = pd.read_csv('Results/SOM results/hpv_positive_som_top_direct_drug_candidates_agg.csv')
hpv_positive_som_direct_drug_candidates['MUT_TYPE'] = 'SOMATIC'
hpv_positive_som_indirect_drug_candidates = pd.read_csv('Results/SOM results/hpv_positive_som_top_indirect_drug_candidates_agg.csv')
hpv_positive_som_indirect_drug_candidates['MUT_TYPE'] = 'SOMATIC'

hpv_negative_som_genes = pd.read_csv('Results/SOM results/HPV negative top genes.csv')
hpv_negative_som_direct_drug_candidates = pd.read_csv('Results/SOM results/hpv_negative_som_top_direct_drug_candidates_agg.csv')
hpv_negative_som_direct_drug_candidates['MUT_TYPE'] = 'SOMATIC'
hpv_negative_som_indirect_drug_candidates = pd.read_csv('Results/SOM results/hpv_negative_som_top_indirect_drug_candidates_agg.csv')
hpv_negative_som_indirect_drug_candidates['MUT_TYPE'] = 'SOMATIC'

#### Overlap

##### HPV+

In [11]:
### number of unique drugs of all hpv positive both direct and indirect
num_unique_pos_drugs = len(list(set(list(set(hpv_positive_direct_drug_candidates['DRUG'].str.lower()))
                           + list(set(hpv_positive_indirect_drug_candidates['DRUG'].str.lower())) 
                           + list(set(hpv_positive_som_direct_drug_candidates['DRUG'].str.lower())) 
                           + list(set(hpv_positive_som_indirect_drug_candidates['DRUG'].str.lower())))))
num_unique_pos_drugs

213

In [12]:
# Get the set of unique HPV positive direct drug candidates
hpv_positive_direct_drugs = set(hpv_positive_direct_drug_candidates['DRUG'].str.lower())
hpv_positive_som_direct_drugs = set(hpv_positive_som_direct_drug_candidates['DRUG'].str.lower())

# Combine both sets to get all unique HPV positive direct drug candidates
all_hpv_positive_direct_drugs = hpv_positive_direct_drugs.union(hpv_positive_som_direct_drugs)

print(f"Number of unique HPV positive direct drug candidates: {len(all_hpv_positive_direct_drugs)}")
print("\nHPV positive direct drug candidates:")
print(sorted(all_hpv_positive_direct_drugs))

Number of unique HPV positive direct drug candidates: 15

HPV positive direct drug candidates:
['buparlisib', 'ch-5132799', 'cladribine', 'copanlisib', 'copper', 'golotimod', 'nadh', 'nelarabine', 'tg-100801', 'wortmannin', 'xl765', 'zinc', 'zinc acetate', 'zinc chloride', 'zinc sulfate, unspecified form']


In [13]:
# Get the set of unique HPV positive indirect drug candidates
hpv_positive_indirect_drugs = set(hpv_positive_indirect_drug_candidates['DRUG'].str.lower())
hpv_positive_som_indirect_drugs = set(hpv_positive_som_indirect_drug_candidates['DRUG'].str.lower())

# Combine both sets to get all unique HPV positive indirect drug candidates
all_hpv_positive_indirect_drugs = hpv_positive_indirect_drugs.union(hpv_positive_som_indirect_drugs)

print(f"Number of unique HPV positive indirect drug candidates: {len(all_hpv_positive_indirect_drugs)}")
print("\nHPV positive indirect drug candidates:")
print(sorted(all_hpv_positive_indirect_drugs))

Number of unique HPV positive indirect drug candidates: 207

HPV positive indirect drug candidates:
['1-chloro-6-(4-hydroxyphenyl)-2-naphthol', '2-chloro-5-nitro-n-phenylbenzamide', '2-tert-butyl-9-fluoro-1,6-dihydrobenzo[h]imidazo[4,5-f]isoquinolin-7-one', '4-(4-methoxy-1h-pyrrolo[2,3-b]pyridin-3-yl)pyrimidin-2-amine', '4-(4-propoxy-1h-pyrrolo[2,3-b]pyridin-3-yl)pyrimidin-2-amine', '9,9,9-trifluoro-8-oxo-n-phenylnonanamide', 'abrocitinib', 'aceclidine', 'acetylsalicylic acid', 'acitretin', 'aclidinium', 'afatinib', 'ag-24322', 'aleglitazar', 'alteplase', 'altiratinib', 'alvocidib', 'amuvatinib', 'an-9', 'anisotropine methylbromide', 'aripiprazole', 'aripiprazole lauroxil', 'arsenic trioxide', 'artenimol', 'axitinib', 'baricitinib', 'benzquinamide', 'bethanechol', 'bezafibrate', 'bimiralisib', 'bisindolylmaleimide i', 'bms-690514', 'bms-754807', 'bosutinib', 'brigatinib', 'brincidofovir', 'brompheniramine', 'buparlisib', 'cabozantinib', 'canertinib', 'capivasertib', 'cerdulatinib', 'ch

In [14]:
# Get the set of unique HPV positive direct drug candidates
hpv_positive_direct_drugs = set(hpv_positive_direct_drug_candidates['DRUG'].str.lower())
hpv_positive_som_direct_drugs = set(hpv_positive_som_direct_drug_candidates['DRUG'].str.lower())

# Get the set of unique HPV positive indirect drug candidates
hpv_positive_indirect_drugs = set(hpv_positive_indirect_drug_candidates['DRUG'].str.lower())
hpv_positive_som_indirect_drugs = set(hpv_positive_som_indirect_drug_candidates['DRUG'].str.lower())

# Combine both sets to get all unique HPV positive direct and indirect drug candidates
all_hpv_positive_direct_drugs = hpv_positive_direct_drugs.union(hpv_positive_som_direct_drugs)
all_hpv_positive_indirect_drugs = hpv_positive_indirect_drugs.union(hpv_positive_som_indirect_drugs)


# Find the overlap between direct and indirect drug candidates
overlapping_drugs = all_hpv_positive_direct_drugs.intersection(all_hpv_positive_indirect_drugs)

print(f"Number of unique HPV positive direct drug candidates: {len(all_hpv_positive_direct_drugs)}")
print(f"Number of unique HPV positive indirect drug candidates: {len(all_hpv_positive_indirect_drugs)}")
print(f"Number of overlapping drugs between direct and indirect: {len(overlapping_drugs)}")
print("\nOverlapping drugs:")
print(sorted(overlapping_drugs))

Number of unique HPV positive direct drug candidates: 15
Number of unique HPV positive indirect drug candidates: 207
Number of overlapping drugs between direct and indirect: 9

Overlapping drugs:
['buparlisib', 'ch-5132799', 'cladribine', 'copanlisib', 'golotimod', 'nadh', 'nelarabine', 'tg-100801', 'wortmannin']


##### HPV-

In [15]:
### number of unique drugs of all hpv negative both direct and indirect
num_unique_neg_drugs = len(list(set(list(set(hpv_negative_direct_drug_candidates['DRUG'].str.lower()))
                           + list(set(hpv_negative_indirect_drug_candidates['DRUG'].str.lower())) 
                           + list(set(hpv_negative_som_direct_drug_candidates['DRUG'].str.lower())) 
                           + list(set(hpv_negative_som_indirect_drug_candidates['DRUG'].str.lower())))))
num_unique_neg_drugs

132

In [16]:
## HPV- Drug Candidates
### Number of unique direct drug candidates
# Get the set of unique HPV negative direct drug candidates
hpv_negative_direct_drugs = set(hpv_negative_direct_drug_candidates['DRUG'].str.lower())
hpv_negative_som_direct_drugs = set(hpv_negative_som_direct_drug_candidates['DRUG'].str.lower())

# Combine both sets to get all unique HPV negative direct drug candidates
all_hpv_negative_direct_drugs = hpv_negative_direct_drugs.union(hpv_negative_som_direct_drugs)

print(f"Number of unique HPV negative direct drug candidates: {len(all_hpv_negative_direct_drugs)}")
print("\nHPV negative direct drug candidates:")
print(sorted(all_hpv_negative_direct_drugs))


Number of unique HPV negative direct drug candidates: 28

HPV negative direct drug candidates:
['acetylsalicylic acid', 'biotin', 'bisindolylmaleimide i', 'brexanolone', 'bryostatin 1', 'caffeine', 'carisoprodol', 'copper', 'dasatinib', 'fludiazepam', 'foreskin fibroblast (neonatal)', 'foreskin keratinocyte (neonatal)', 'fostamatinib', 'gamma-aminobutyric acid', 'heparin', 'meprobamate', 'metharbital', 'nadh', 'regorafenib', 'secobarbital', 'talbutal', 'wortmannin', 'xl765', 'zinc', 'zinc acetate', 'zinc chloride', 'zinc sulfate, unspecified form', 'zolpidem']


In [17]:
# Get the set of unique HPV negative indirect drug candidates
hpv_negative_indirect_drugs = set(hpv_negative_indirect_drug_candidates['DRUG'].str.lower())
hpv_negative_som_indirect_drugs = set(hpv_negative_som_indirect_drug_candidates['DRUG'].str.lower())

# Combine both sets to get all unique HPV negative indirect drug candidates
all_hpv_negative_indirect_drugs = hpv_negative_indirect_drugs.union(hpv_negative_som_indirect_drugs)

print(f"Number of unique HPV negative indirect drug candidates: {len(all_hpv_negative_indirect_drugs)}")
print("\nHPV negative indirect drug candidates:")
print(sorted(all_hpv_negative_indirect_drugs))

Number of unique HPV negative indirect drug candidates: 126

HPV negative indirect drug candidates:
['(7s)-2-(2-aminopyrimidin-4-yl)-7-(2-fluoroethyl)-1,5,6,7-tetrahydro-4h-pyrrolo[3,2-c]pyridin-4-one', '3-isobutyl-1-methyl-7h-xanthine', 'aceclidine', 'acetylsalicylic acid', 'aclidinium', 'ag-24322', 'alsterpaullone', 'altiratinib', 'alvocidib', 'amuvatinib', 'antithymocyte immunoglobulin (rabbit)', 'aripiprazole lauroxil', 'arsenic trioxide', 'artenimol', 'avagacestat', 'begacestat', 'bethanechol', 'biotin', 'bisindolylmaleimide i', 'bms-690514', 'bosutinib', 'brexanolone', 'brigatinib', 'bryostatin 1', 'caffeine', 'calcium citrate', 'calcium phosphate', 'canertinib', 'carfilzomib', 'carisoprodol', 'chlorprothixene', 'cholic acid', 'ci-1040', 'clozapine', 'conestat alfa', 'darifenacin', 'dasatinib', 'decitabine', 'e-2012', 'enzastaurin', 'erdafitinib', 'esflurbiprofen', 'famitinib', 'fesoterodine', 'fg-9041', 'fludiazepam', 'fn-1501', 'foreskin fibroblast (neonatal)', 'foreskin kerati

In [18]:
# Get the set of unique HPV negative drug candidates (both direct and indirect)
hpv_negative_direct_drugs = set(hpv_negative_direct_drug_candidates['DRUG'].str.lower())
hpv_negative_som_direct_drugs = set(hpv_negative_som_direct_drug_candidates['DRUG'].str.lower())
hpv_negative_indirect_drugs = set(hpv_negative_indirect_drug_candidates['DRUG'].str.lower())
hpv_negative_som_indirect_drugs = set(hpv_negative_som_indirect_drug_candidates['DRUG'].str.lower())

# Combine all HPV negative drug sets
all_hpv_negative_direct_drugs = hpv_negative_direct_drugs.union(hpv_negative_som_direct_drugs)
all_hpv_negative_direct_drugs = set(drug.lower() for drug in all_hpv_negative_direct_drugs)
all_hpv_negative_indirect_drugs = hpv_negative_indirect_drugs.union(hpv_negative_som_indirect_drugs)
all_hpv_negative_indirect_drugs = set(drug.lower() for drug in all_hpv_negative_indirect_drugs)

# Find the overlap between direct and indirect drug candidates for HPV negative
overlapping_drugs = all_hpv_negative_direct_drugs.intersection(all_hpv_negative_indirect_drugs)


print(f"Number of unique HPV negative direct drug candidates: {len(all_hpv_negative_direct_drugs)}")
print(f"Number of unique HPV negative indirect drug candidates: {len(all_hpv_negative_indirect_drugs)}")
print(f"Number of overlapping drugs between direct and indirect: {len(overlapping_drugs)}")
print("\nOverlapping drugs:")
print(sorted(overlapping_drugs))

Number of unique HPV negative direct drug candidates: 28
Number of unique HPV negative indirect drug candidates: 126
Number of overlapping drugs between direct and indirect: 22

Overlapping drugs:
['acetylsalicylic acid', 'biotin', 'bisindolylmaleimide i', 'brexanolone', 'bryostatin 1', 'caffeine', 'carisoprodol', 'dasatinib', 'fludiazepam', 'foreskin fibroblast (neonatal)', 'foreskin keratinocyte (neonatal)', 'fostamatinib', 'gamma-aminobutyric acid', 'heparin', 'meprobamate', 'metharbital', 'nadh', 'regorafenib', 'secobarbital', 'talbutal', 'wortmannin', 'zolpidem']


#### Literature results

In [19]:
extracted_target_df= pd.read_csv('Validation pipeline/Results/cleaned_extracted_targets_all_pub_after_2000_GPU_2b_gemma.csv')
extracted_target_df_combined = pd.read_csv('Validation pipeline/Results/cleaned_extracted_combined_targets_all_pub_after_2000_GPU_2b_gemma.csv')

In [20]:
### accumulate all genes available in drugbank or ppi
Drug_bank_genes = list(Drug_bank['gene'].values)
ppi_genes = list(protein_interaction['Translated_protein_1'].values)
all_ppi_drugbank = list(set(Drug_bank_genes + ppi_genes))

## HPV+

#### Genes

In [21]:
hpv_positive_som_genes

,Gene,Count,Cohort_Frequency,Normalized_Count,Normalized_Cohort_Frequency,P_Value,Adjusted_P_Value,Significant,Empirical_P_Value,Adjusted_Empirical_P_Value,frequency_percentage,mutation_score
0,PIK3CA,18,17,0.005473,0.005169,4.738394e-19,2.079207e-15,True,0.0001,0.036563,23.611111,0.129219
1,ZNF750,11,8,0.005071,0.003688,7.324983e-12,1.607101e-08,True,0.0001,0.036563,11.111111,0.056350
2,CYLD,8,8,0.002677,0.002677,6.578872e-07,9.622697e-04,True,0.0001,0.036563,11.111111,0.029749
3,EP300,9,9,0.001243,0.001243,6.021135e-05,4.723959e-02,True,0.0001,0.036563,12.500000,0.015534
4,CCDC191,6,6,0.002083,0.002083,6.589348e-05,4.723959e-02,True,0.0001,0.036563,8.333333,0.017355
5,LRRC37B,6,5,0.001994,0.001662,8.343138e-05,4.723959e-02,True,0.0001,0.036563,6.944444,0.013847


In [22]:
### combine hpv positive somatic genes and cnv genes
hpv_positive_som_genes['MUT_TYPE'] = 'SOMATIC'
hpv_positive_som_genes['gene_name'] = hpv_positive_som_genes['Gene']
hpv_positive_som_genes['q_value']= hpv_positive_som_genes['Adjusted_P_Value']
hpv_positive_som_genes['empirical_q_value'] = hpv_positive_som_genes['Adjusted_Empirical_P_Value']
hpv_positive_combined_genes = pd.concat([hpv_positive_genes, hpv_positive_som_genes], axis=0)
### aggregate by GENE to get unique genes with both mutation types
hpv_positive_combined_genes['gene_name'] = hpv_positive_combined_genes['gene_name'].str.upper()
extracted_target_df_combined['GENE'] = extracted_target_df_combined['GENE'].str.upper()
hpv_positive_combined_genes = hpv_positive_combined_genes.groupby('gene_name').agg({
    'MUT_TYPE': lambda x: ', '.join(x),
    'q_value': lambda x: ', '.join(x.astype(str)) if len(x) > 1 else x.iloc[0].astype(str),
    'empirical_q_value': lambda x: ', '.join(x.astype(str)) if len(x) > 1 else x.iloc[0].astype(str)
}).reset_index()

hpv_positive_combined_genes


,gene_name,MUT_TYPE,q_value,empirical_q_value
0,ACE2,DELETION,8.234936784924659e-31,0.0203366143594484
1,ACTL6A,AMPLIFICATION,4.9013845356499375e-54,0.0116600765426333
2,ADIPOQ,AMPLIFICATION,4.9013845356499375e-54,0.0116600765426333
3,AHSG,AMPLIFICATION,4.9013845356499375e-54,0.0116600765426333
4,ANOS1,DELETION,8.234936784924659e-31,0.0203366143594484
5,ATXN3L,DELETION,8.234936784924659e-31,0.0203366143594484
6,BCL6,AMPLIFICATION,1.1060251600493612e-54,0.0116600765426333
7,BMX,DELETION,8.234936784924659e-31,0.0203366143594484
8,CCDC191,SOMATIC,0.0472395868080887,0.0365630103656301
9,CDKL5,DELETION,8.234936784924659e-31,0.0203366143594484


In [23]:
len(set(hpv_positive_combined_genes['gene_name']))

60

In [24]:
### merge genes with number of articles, pubmed id from literature data
hpv_positive_genes_with_lit = pd.merge(hpv_positive_combined_genes, extracted_target_df_combined, how = 'left', left_on='gene_name', right_on='GENE')
hpv_positive_genes_with_lit.drop(columns =['INDEX'], inplace = True)

In [25]:
hpv_positive_genes_with_lit[hpv_positive_genes_with_lit['NUMBER_OF_ARTICLES']>0]

,gene_name,MUT_TYPE,q_value,empirical_q_value,GENE,PMID,NUMBER_OF_ARTICLES
6,BCL6,AMPLIFICATION,1.1060251600493612e-54,0.0116600765426333,BCL6,"11224600, 11420458, 14685876, 17429099",4.0
10,CLDN1,AMPLIFICATION,6.5911053664735535e-53,0.0116600765426333,CLDN1,"15170668, 17091452",2.0
12,CYLD,SOMATIC,0.0009622697492491,0.0365630103656301,CYLD,"16900776, 18497946",2.0
37,PIK3CA,"AMPLIFICATION, SOMATIC","4.9013845356499375e-54, 2.079207200788839e-15","0.0116600765426333, 0.0365630103656301",PIK3CA,"11358835, 11836556, 11959846, 14581353, 155436...",17.0
42,RFC4,AMPLIFICATION,4.9013845356499375e-54,0.0116600765426333,RFC4,16467079,1.0
49,SOX2,AMPLIFICATION,1.2920437544143294e-55,0.0116600765426333,SOX2,15942670,1.0
52,TLR7,DELETION,8.234936784924659e-31,0.0203366143594484,TLR7,17201162,1.0


In [26]:
hpv_positive_genes_with_lit = hpv_positive_genes_with_lit[hpv_positive_genes_with_lit['NUMBER_OF_ARTICLES']>0]

In [27]:
hpv_positive_genes_with_lit.to_csv('Results/HPV positive gene results.csv')

In [28]:
### export to final results output
hpv_positive_genes_with_lit.to_csv('Results/Final Results/HPV Positive validated genes.csv')

#### Direct

In [29]:
### merge all hpv postive direct drug candidates
hpv_positive_final_direct = pd.concat([hpv_positive_direct_drug_candidates, hpv_positive_som_direct_drug_candidates])
### group by drug and comma seperate genes and mutation type
### columns: DRUG	GENE_TARGET	NUM_DIRECT_TARGETS_HIT	TOTAL_TARGETS_IN_DRUGBANK	PERCENTAGE_OF_TARGETS_HIT	GENE_GISTIC	GENE_normalized_gistic_score	
# ACTION	SPECIFIC_FUNCTION	drug_hypergeom_p_value	drug_hypergeom_fdr	drug_empirical_p_value	drug_empirical_fdr	MUT_TYPE	GENE_Cohort_Frequency	
# GENE_Normalized_Count	GENE_Normalized_Cohort_Frequency	GENE_SIGNIFICANT
hpv_positive_final_direct = hpv_positive_final_direct.groupby('DRUG').agg({'GENE_TARGET': lambda x: ', '.join(x),
                                               'MUT_TYPE': lambda x: ', '.join(x),
                                                  'NUM_DIRECT_TARGETS_HIT': 'first',
                                                    'TOTAL_TARGETS_IN_DRUGBANK': 'first',
                                                    'PERCENTAGE_OF_TARGETS_HIT': 'first',
                                                    'ACTION': 'first',
                                                    'SPECIFIC_FUNCTION': 'first',
                                                    'drug_hypergeom_p_value': 'first',
                                                    'drug_hypergeom_fdr': 'first',
                                                    'drug_empirical_p_value': 'first',
                                                    'drug_empirical_fdr': 'first',
                                                    }).reset_index()

In [30]:
hpv_positive_final_direct

,DRUG,GENE_TARGET,MUT_TYPE,NUM_DIRECT_TARGETS_HIT,TOTAL_TARGETS_IN_DRUGBANK,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_p_value,drug_hypergeom_fdr,drug_empirical_p_value,drug_empirical_fdr
0,Buparlisib,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,4,25.000000,inhibitor,1-phosphatidylinositol-3-kinase activity,5.173220e-04,3.304624e-02,0.00059,0.038789
1,CH-5132799,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,4,25.000000,inhibitor,1-phosphatidylinositol-3-kinase activity,5.173220e-04,3.304624e-02,0.00059,0.038789
2,Cladribine,POLA1,DELETION,1,12,8.333333,inhibitor,chromatin binding,4.242621e-05,1.528532e-02,0.00006,0.021617
3,Copanlisib,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,4,25.000000,inhibitor,1-phosphatidylinositol-3-kinase activity,5.173220e-04,3.304624e-02,0.00059,0.038789
4,Copper,"AHSG, KNG1","AMPLIFICATION, AMPLIFICATION",2,146,1.369863,None,cysteine-type endopeptidase inhibitor activity,8.310495e-15,2.495088e-11,0.00001,0.002095
5,Golotimod,TLR7,DELETION,1,5,20.000000,None,double-stranded RNA binding,1.607473e-05,6.295005e-03,0.00004,0.015664
6,NADH,NDUFB5,AMPLIFICATION,1,144,0.694444,None,NADH dehydrogenase (ubiquinone) activity,6.252143e-08,2.252522e-05,0.00001,0.002095
7,Nelarabine,POLA1,DELETION,1,5,20.000000,inhibitor,chromatin binding,1.467616e-04,2.937515e-02,0.00017,0.036457
8,TG-100801,VEGFD,DELETION,1,8,12.500000,inhibitor,chemoattractant activity,4.043797e-05,1.517603e-02,0.00008,0.025734
9,Wortmannin,PIK3CA,SOMATIC,1,5,20.000000,None,1-phosphatidylinositol-3-kinase activity,4.559776e-05,5.475987e-03,0.00005,0.006623


In [31]:
extracted_target_df_combined

,GENE,PMID,INDEX,NUMBER_OF_ARTICLES
0,000-2,"11302242, 11302242","2610, 2610",1
1,10,"12608845, 12768769, 14967420, 15193028, 180565...","15288, 17016, 27005, 29481, 57658, 61105",6
2,106PRE,18186293,58737,1
3,106R,18186293,58737,1
4,106RECR,18186293,58737,1
...,...,...,...,...
6072,ZP-V3,12464647,13458,1
6073,ZP-V4,12464647,13458,1
6074,ZYGOMA,15883929,36459,1
6075,ZYGOMATIC,"12775236, 17522494","17081, 53057",2


In [32]:
#### add columns to hpv_positive_final_direct for PMIds and NUMBER_OF_ARTICLES from extracted_target_df_combined
### ADD COLUMNS: PMIDs, NUMBER_OF_ARTICLES, gene
### combine based on gene target, and if any of the genes in GENE_TARGET are in extracted_target_df_combined, then add the PMIDs and NUMBER_OF_ARTICLES

hpv_positive_final_direct['PMID'] = ''
hpv_positive_final_direct['NUMBER_OF_ARTICLES'] = 0
hpv_positive_final_direct['LITERATURE_GENE_TARGETS'] = ''
for index, row in hpv_positive_final_direct.iterrows():
    gene_targets = row['GENE_TARGET'].split(', ')
    gene_targets = [gene.strip() for gene in gene_targets]
    gene_targets = list(set(gene_targets))
    pmids_set = set()
    literature_gene_targets = set()
    number_of_articles = 0
    for gene in gene_targets:
        matched_rows = extracted_target_df_combined[extracted_target_df_combined['GENE'] == gene]
        for _, matched_row in matched_rows.iterrows():
            pmids = matched_row['PMID'].split(', ')
            pmids_set.update(pmids)
            number_of_articles += matched_row['NUMBER_OF_ARTICLES']
            literature_gene_targets.add(matched_row['GENE'])
    hpv_positive_final_direct.at[index, 'LITERATURE_GENE_TARGETS'] = ', '.join(list(set(literature_gene_targets)))
    hpv_positive_final_direct.at[index, 'PMID'] = ', '.join(pmids_set)
    hpv_positive_final_direct.at[index, 'NUMBER_OF_ARTICLES'] = number_of_articles

In [33]:
hpv_positive_final_direct[hpv_positive_final_direct['NUMBER_OF_ARTICLES'] > 0]

,DRUG,GENE_TARGET,MUT_TYPE,NUM_DIRECT_TARGETS_HIT,TOTAL_TARGETS_IN_DRUGBANK,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_p_value,drug_hypergeom_fdr,drug_empirical_p_value,drug_empirical_fdr,PMID,NUMBER_OF_ARTICLES,LITERATURE_GENE_TARGETS
0,Buparlisib,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,4,25.0,inhibitor,1-phosphatidylinositol-3-kinase activity,0.000517,0.033046,0.00059,0.038789,"11959846, 15700036, 16807070, 17990317, 166763...",17,PIK3CA
1,CH-5132799,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,4,25.0,inhibitor,1-phosphatidylinositol-3-kinase activity,0.000517,0.033046,0.00059,0.038789,"11959846, 15700036, 16807070, 17990317, 166763...",17,PIK3CA
3,Copanlisib,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,4,25.0,inhibitor,1-phosphatidylinositol-3-kinase activity,0.000517,0.033046,0.00059,0.038789,"11959846, 15700036, 16807070, 17990317, 166763...",17,PIK3CA
5,Golotimod,TLR7,DELETION,1,5,20.0,None,double-stranded RNA binding,0.000016,0.006295,0.00004,0.015664,17201162,1,TLR7
9,Wortmannin,PIK3CA,SOMATIC,1,5,20.0,None,1-phosphatidylinositol-3-kinase activity,0.000046,0.005476,0.00005,0.006623,"11959846, 15700036, 16807070, 17990317, 166763...",17,PIK3CA
10,XL765,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,5,20.0,None,1-phosphatidylinositol-3-kinase activity,0.000078,0.008239,0.00012,0.012568,"11959846, 15700036, 16807070, 17990317, 166763...",17,PIK3CA


In [34]:
### ensure that only drugs with NUMBER_OF_ARTICLES > 0 are saved, so that they have literature support
hpv_positive_final_direct = hpv_positive_final_direct[hpv_positive_final_direct['NUMBER_OF_ARTICLES'] > 0]
### save results
hpv_positive_final_direct.to_csv('Results/HPV Positive direct results.csv')

#### Indirect

In [35]:
### merge all hpv positive indirect drug candidates
hpv_positive_final_indirect = pd.concat([hpv_positive_indirect_drug_candidates, hpv_positive_som_indirect_drug_candidates], ignore_index=True)
hpv_positive_final_indirect['ACTION'] = hpv_positive_final_indirect['ACTION'].fillna('UNKNOWN')
hpv_positive_final_indirect['SPECIFIC_FUNCTION'] = hpv_positive_final_indirect['SPECIFIC_FUNCTION'].fillna('UNKNOWN')
hpv_positive_final_indirect['drug_hypergeom_fdr'] = hpv_positive_final_indirect['drug_hypergeom_fdr'].fillna('UNKNOWN')
hpv_positive_final_indirect['drug_empirical_fdr'] = hpv_positive_final_indirect['drug_empirical_fdr'].fillna('UNKNOWN')
hpv_positive_final_indirect['MUT_TYPE'] = hpv_positive_final_indirect['MUT_TYPE'].fillna('UNKNOWN')

### aggregate/group by drug name
### columns: DRUG	CONNECTED_TO (risk gene)	
# Number of risk or immediate neighbor genes
# targeted	total_genes_targeted_in_drugbank	
# PERCENTAGE_OF_TARGETS_HIT
# Number of indirect genes connected to this risk gene	
# GENE_TARGET	GENE_Cohort_Frequency	
# GENE_Normalized_Count	GENE_Normalized_Cohort_Frequency	
# ACTION	SPECIFIC_FUNCTION	drug_hypergeom_fdr	drug_empirical_fdr

hpv_positive_final_indirect = hpv_positive_final_indirect.groupby(['DRUG']).agg({
    'GENE_TARGET': lambda x: ', '.join(x),
    'CONNECTED_TO (risk gene)': lambda x: ', '.join(x),
    'Number of risk or immediate neighbor genes targeted': 'first',
    'total_genes_targeted_in_drugbank': 'first',
    'PERCENTAGE_OF_TARGETS_HIT': 'first',
    'ACTION': lambda x: ', '.join(x),
    'SPECIFIC_FUNCTION': lambda x: ', '.join(x),
    'drug_hypergeom_fdr': 'max',
    'drug_empirical_fdr': 'max',
    'MUT_TYPE': lambda x: ', '.join(x)
}).reset_index()
hpv_positive_final_indirect.sort_values(by = 'drug_empirical_fdr', ascending = True).head(25)

,DRUG,GENE_TARGET,CONNECTED_TO (risk gene),Number of risk or immediate neighbor genes targeted,total_genes_targeted_in_drugbank,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_fdr,drug_empirical_fdr,MUT_TYPE
169,somatostatin,"SSTR4, SSTR3, OPRM1, SSTR5, SSTR1, SSTR2,OPRD1...","SST,KNG1,GNB4",7,7,100.000000,"agonist, inhibitor","neuropeptide binding, G protein-coupled recept...",4.376727e-04,0.002095,AMPLIFICATION
91,human c1-esterase inhibitor,"C1R, C1S,F2,PLAT, F12, F2,F2, KLKB1, F12, C1S,...","MASP1,AHSG,HRG,KNG1",7,7,100.000000,inhibitor,calcium ion binding,4.376727e-04,0.002095,AMPLIFICATION
190,trihexyphenidyl,"CHRM2, CHRM1,ADRB3,CHRM1, CHRM3, CHRM4, CHRM5,...","SST,ADIPOQ,KNG1,GNB4",6,6,100.000000,antagonist,"arrestin family protein binding, G protein-cou...",2.145256e-03,0.002095,AMPLIFICATION
117,nadh,"NDUFA8, NDUFS5, NDUFS2, NDUFA2, NDUFB10, NDUFV...",NDUFB5,46,144,31.944444,binder,"NADH dehydrogenase (ubiquinone) activity, 4 ir...",2.252522e-05,0.002095,AMPLIFICATION
44,ci-1040,"AKT1,LCK, RPS6KB1, AKT1, MAPK1,AKT1,RPS6KB1,CH...","SOX2,PIK3CA,ADIPOQ,EIF4A2,RFC4,GNB4,MRPL47",10,15,66.666667,inhibitor,14-3-3 protein binding,1.910923e-03,0.002095,AMPLIFICATION
23,artenimol,"GAPDH, GAPDH,NPM1,ALB,ANXA2, ALB,CCT3, HSPA8, ...","SOX2,PIK3CA,ADIPOQ,AHSG,DNAJB11,EIF4A2,FXR1,HR...",36,104,34.615385,ligand,aspartic-type endopeptidase inhibitor activity...,4.922768e-05,0.002095,"DELETION,AMPLIFICATION"
47,conestat alfa,"C1R, C1S,F2,PLAT, F12, F2,F2, KLKB1, F12, C1S,...","MASP1,AHSG,HRG,KNG1",7,7,100.000000,inhibitor,calcium ion binding,4.376727e-04,0.002095,AMPLIFICATION
21,aripiprazole lauroxil,"HTR1B, DRD2, HTR1E, CHRM2, HTR1A, HTR1D, CHRM1...","SST,GNB4",13,25,52.000000,partial agonist,"G protein-coupled serotonin receptor activity,...",3.006270e-03,0.002095,AMPLIFICATION
152,quetiapine,"HTR1B, DRD2, HTR1E, CHRM2, HTR1A, HTR1D, CHRM1...","SST,KNG1,GNB4",13,24,54.166667,"ligand, antagonist","G protein-coupled serotonin receptor activity,...",2.029951e-03,0.002095,AMPLIFICATION
153,regorafenib,"RAF1, FGFR1, KDR, PDGFRB, FLT1, BRAF, KIT, RET...","PIK3CA,GNB4,CLDN1,IL1RAP,ANOS1,CNKSR2,RPS6KA3,...",15,18,83.333333,"inhibitor, inhibitor","ATP binding, actin filament binding, ATP bindi...",2.845495e-07,0.002309,"DELETION,AMPLIFICATION, SOMATIC"


In [36]:
# hpv_positive_final_indirect[hpv_positive_final_indirect['DRUG'].str.lower().isin(hpv_positive_final_direct['DRUG'].str.lower())]

In [37]:
### extract the unique number of GENE_TARGETs from , seperated list in each of the GENE_TARGET column
def extract_unique_gene_targets(df, column_name):
    return len(set([gene.strip() for sublist in df[column_name].dropna().str.split(',') for gene in sublist]))

hpv_pos_indirect_genes = extract_unique_gene_targets(hpv_positive_indirect_drug_candidates, 'GENE_TARGET')
print(hpv_pos_indirect_genes)

277


In [38]:
#### add columns to hpv_positive_final_direct for PMIds and NUMBER_OF_ARTICLES from extracted_target_df_combined
### ADD COLUMNS: PMIDs, NUMBER_OF_ARTICLES, gene
### combine based on gene target, and if any of the genes in GENE_TARGET are in extracted_target_df_combined, then add the PMIDs and NUMBER_OF_ARTICLES
hpv_positive_final_indirect['PMID'] = ''
hpv_positive_final_indirect['NUMBER_OF_ARTICLES'] = 0
hpv_positive_final_indirect['LITERATURE_GENE_TARGETS'] = ''
for index, row in hpv_positive_final_indirect.iterrows():
    gene_targets = row['GENE_TARGET'].split(',')
    gene_targets = [gene.strip() for gene in gene_targets]
    pmids_set = set()
    literature_gene_targets = set()
    number_of_articles = 0
    for gene in gene_targets:
        matched_rows = extracted_target_df_combined[extracted_target_df_combined['GENE'] == gene]
        for _, matched_row in matched_rows.iterrows():
            pmids = matched_row['PMID'].split(',')
            pmids = [pmid.strip() for pmid in pmids]
            pmids_set.update(pmids)
            number_of_articles += matched_row['NUMBER_OF_ARTICLES']
            literature_gene_targets.add(matched_row['GENE'])
    
    hpv_positive_final_indirect.at[index, 'LITERATURE_GENE_TARGETS'] = ', '.join(literature_gene_targets)
    hpv_positive_final_indirect.at[index, 'PMID'] = ', '.join(pmids_set)
    hpv_positive_final_indirect.at[index, 'NUMBER_OF_ARTICLES'] = number_of_articles

### validate risk genes
hpv_positive_final_indirect = hpv_positive_final_indirect[hpv_positive_final_indirect['NUMBER_OF_ARTICLES'] > 0]
hpv_positive_final_indirect['RISK_GENE_PMID'] = ''
hpv_positive_final_indirect['RISK_GENE_NUMBER_OF_ARTICLES'] = 0
hpv_positive_final_indirect['RISK_GENE_LITERATURE_GENE_TARGETS'] = ''
for index, row in hpv_positive_final_indirect.iterrows():
    risk_genes = row['CONNECTED_TO (risk gene)'].split(',')
    risk_genes = [gene.strip() for gene in risk_genes]
    pmids_set = set()
    literature_gene_targets = set()
    number_of_articles = 0
    for gene in risk_genes:
        #print(gene)
        matched_rows = extracted_target_df_combined[extracted_target_df_combined['GENE'] == gene]
        for _, matched_row in matched_rows.iterrows():
            pmids = matched_row['PMID'].split(',')
            pmids = [pmid.strip() for pmid in pmids]
            pmids_set.update(pmids)
            number_of_articles += matched_row['NUMBER_OF_ARTICLES']
            literature_gene_targets.add(matched_row['GENE'])
    
    hpv_positive_final_indirect.at[index, 'RISK_GENE_LITERATURE_GENE_TARGETS'] = ', '.join(literature_gene_targets)
    hpv_positive_final_indirect.at[index, 'RISK_GENE_PMID'] = ', '.join(pmids_set)
    hpv_positive_final_indirect.at[index, 'RISK_GENE_NUMBER_OF_ARTICLES'] = number_of_articles

### ensure that only drugs with NUMBER_OF_ARTICLES > 0 for both drug targets and risk genes are saved, so that they have literature support
hpv_positive_final_indirect = hpv_positive_final_indirect[hpv_positive_final_indirect['NUMBER_OF_ARTICLES'] > 0]
hpv_positive_final_indirect = hpv_positive_final_indirect[hpv_positive_final_indirect['RISK_GENE_NUMBER_OF_ARTICLES'] > 0]

In [39]:
### extract the unique number of GENE_TARGETs from , seperated list in each of the GENE_TARGET column
def extract_unique_gene_targets(df, column_name):
    return len(set([gene.strip() for sublist in df[column_name].dropna().str.split(',') for gene in sublist]))

hpv_pos_final_indirect_genes = extract_unique_gene_targets(hpv_positive_final_indirect, 'GENE_TARGET')
print(hpv_pos_final_indirect_genes)


208


In [40]:
hpv_positive_final_indirect.to_csv('Results/HPV Positive indirect results.csv', index=False)

In [41]:
### extract the unique number of GENE_TARGETs from , seperated list in each of the GENE_TARGET column
def extract_unique_gene_targets(df, column_name):
    return len(set([gene.strip() for sublist in df[column_name].dropna().str.split(',') for gene in sublist]))

hpv_pos_final_indirect_genes = extract_unique_gene_targets(hpv_positive_final_indirect, 'GENE_TARGET')
print(hpv_pos_final_indirect_genes)

208


In [42]:
hpv_positive_final_indirect

,DRUG,GENE_TARGET,CONNECTED_TO (risk gene),Number of risk or immediate neighbor genes targeted,total_genes_targeted_in_drugbank,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_fdr,drug_empirical_fdr,MUT_TYPE,PMID,NUMBER_OF_ARTICLES,LITERATURE_GENE_TARGETS,RISK_GENE_PMID,RISK_GENE_NUMBER_OF_ARTICLES,RISK_GENE_LITERATURE_GENE_TARGETS
0,1-chloro-6-(4-hydroxyphenyl)-2-naphthol,"ESR1, NCOA1, ESR2, ESR1","PIK3CA, EP300",3,3,100.000000,"inhibitor, inhibitor","14-3-3 protein binding, chromatin binding, DNA...",0.012346,0.014723,"SOMATIC, SOMATIC",11309301,2,ESR1,"11959846, 15700036, 16807070, 17990317, 166763...",17,PIK3CA
2,"2-tert-butyl-9-fluoro-1,6-dihydrobenzo[h]imida...","JAK2, TYK2, JAK3, JAK1",PIK3CA,4,5,80.000000,inhibitor,"acetylcholine receptor binding, ATP binding",0.005476,0.008520,SOMATIC,"18204781, 15947106",2,"JAK2, JAK3","11959846, 15700036, 16807070, 17990317, 166763...",17,PIK3CA
5,"9,9,9-trifluoro-8-oxo-n-phenylnonanamide","HDAC6, HDAC10, HDAC4, HDAC1, HDAC6, HDAC2, HDAC8","CYLD, EP300",6,7,85.714286,"inhibitor, inhibitor","actin binding, acetylputrescine deacetylase ac...",0.000066,0.002309,"SOMATIC, SOMATIC","12239610, 16773191",3,"HDAC6, HDAC1","16900776, 18497946",2,CYLD
6,abrocitinib,"JAK2, TYK2, JAK3, JAK1, JAK2, TYK2, JAK3, JAK1","PIK3CA, PIK3CA",4,4,100.000000,"inhibitor, inhibitor","acetylcholine receptor binding, ATP binding, a...",0.033046,0.038601,"AMPLIFICATION, SOMATIC","18204781, 15947106",4,"JAK2, JAK3","11959846, 15700036, 16807070, 17990317, 166763...",34,PIK3CA
8,acetylsalicylic acid,"MYC, TP53, CCND1, MAPK1, TP53, IKBKB, NFKBIA, ...","PIK3CA, CYLD, EP300",7,19,36.842105,"downregulator, inducer, inducer, inhibitor, in...","core promoter sequence-specific DNA binding, 1...",0.005476,0.011401,"SOMATIC, SOMATIC, SOMATIC","18405350, 18075740, 17258495, 15846092, 164905...",200,"PCNA, TP53, NFKBIA, MYC, CCND1","17990317, 15543611, 16676365, 14581353, 169007...",19,"PIK3CA, CYLD"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,ulipristal,"PGR, NR3C1, PGR, AR","PIK3CA, EP300",3,3,100.000000,"modulator, antagonist, modulator","ATPase binding, core promoter sequence-specifi...",0.012346,0.014723,"SOMATIC, SOMATIC","18528886, 16835481, 11556855, 17017191, 166904...",12,"PGR, AR","11959846, 15700036, 16807070, 17990317, 166763...",17,PIK3CA
198,vatalanib,"EGFR,KDR, EGFR, FLT1, FLT4,EGFR,EGFR,FLT1, EGF...","SOX2,PIK3CA,IGF2BP2,SH3KBP1,VEGFD, PIK3CA",4,4,100.000000,"inhibitor, inhibitor","actin filament binding, ATP binding, actin fil...",0.033046,0.038601,"DELETION,AMPLIFICATION, SOMATIC","15833860, 17258495, 15717992, 17121916, 129520...",1776,EGFR,"11959846, 15700036, 16807070, 17990317, 166763...",35,"PIK3CA, SOX2"
199,vorinostat,"HDAC6,HDAC6,HDAC8, HDAC2, HDAC3, HDAC1, HDAC1,...","ATXN3L,OFD1,RBBP7,SCML2,TBL1X, CYLD, EP300",5,6,83.333333,"inhibitor, inhibitor, inhibitor","actin binding, actin binding, core promoter se...",0.027227,0.030023,"DELETION, SOMATIC, SOMATIC","12239610, 16773191",10,"HDAC6, HDAC1","16900776, 18497946",2,CYLD
203,zanubrutinib,"EGFR,JAK2, FGR, BTK, ERBB4, ITK, LCK, EGFR, JA...","SOX2,PIK3CA,IGF2BP2, PIK3CA",10,15,66.666667,"inhibitor, inhibitor","actin filament binding, acetylcholine receptor...",0.001911,0.003833,"AMPLIFICATION, SOMATIC","17208308, 12649172, 11303632, 14581353, 177629...",1246,"ERBB4, ERBB2, JAK2, EGFR, JAK3","11959846, 15700036, 16807070, 17990317, 166763...",35,"PIK3CA, SOX2"


#### overall

In [43]:
hpv_positive_final_direct

,DRUG,GENE_TARGET,MUT_TYPE,NUM_DIRECT_TARGETS_HIT,TOTAL_TARGETS_IN_DRUGBANK,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_p_value,drug_hypergeom_fdr,drug_empirical_p_value,drug_empirical_fdr,PMID,NUMBER_OF_ARTICLES,LITERATURE_GENE_TARGETS
0,Buparlisib,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,4,25.0,inhibitor,1-phosphatidylinositol-3-kinase activity,0.000517,0.033046,0.00059,0.038789,"11959846, 15700036, 16807070, 17990317, 166763...",17,PIK3CA
1,CH-5132799,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,4,25.0,inhibitor,1-phosphatidylinositol-3-kinase activity,0.000517,0.033046,0.00059,0.038789,"11959846, 15700036, 16807070, 17990317, 166763...",17,PIK3CA
3,Copanlisib,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,4,25.0,inhibitor,1-phosphatidylinositol-3-kinase activity,0.000517,0.033046,0.00059,0.038789,"11959846, 15700036, 16807070, 17990317, 166763...",17,PIK3CA
5,Golotimod,TLR7,DELETION,1,5,20.0,None,double-stranded RNA binding,0.000016,0.006295,0.00004,0.015664,17201162,1,TLR7
9,Wortmannin,PIK3CA,SOMATIC,1,5,20.0,None,1-phosphatidylinositol-3-kinase activity,0.000046,0.005476,0.00005,0.006623,"11959846, 15700036, 16807070, 17990317, 166763...",17,PIK3CA
10,XL765,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,5,20.0,None,1-phosphatidylinositol-3-kinase activity,0.000078,0.008239,0.00012,0.012568,"11959846, 15700036, 16807070, 17990317, 166763...",17,PIK3CA


In [44]:
extracted_target_df_combined[extracted_target_df_combined['GENE'] == 'PIK3CA']['PMID']

4484    11358835, 11836556, 11959846, 14581353, 155436...
Name: PMID, dtype: object

In [45]:
### combined direct and indirect final results
### final columns: DRUG   GENE_TARGET CONNECTED_TO (risk gene)	NUM_DIRECT_TARGETS_HIT  
# Number of risk or immediate neighbor genes targeted	TOTAL_TARGETS_IN_DRUGBANK	PERCENTAGE_OF_TARGETS_HIT	
# ACTION	SPECIFIC_FUNCTION	drug_hypergeom_fdr	drug_empirical_fdr	
### column for Target description: direct, or indirect
hpv_positive_final_direct['Target_Description'] = 'Direct'
hpv_positive_final_indirect['Target_Description'] = 'Indirect'
hpv_positive_final_results = pd.concat([hpv_positive_final_direct, hpv_positive_final_indirect], ignore_index=True)
### replace any null with 'NA' in the hwole dataframe
hpv_positive_final_results = hpv_positive_final_results.fillna('NA')
hpv_positive_final_results = hpv_positive_final_results.groupby('DRUG').agg({
    'GENE_TARGET': lambda x: ', '.join(x),
    'CONNECTED_TO (risk gene)': lambda x: ', '.join(x),
    'NUM_DIRECT_TARGETS_HIT': 'first',
    'Number of risk or immediate neighbor genes targeted': 'first',
    'TOTAL_TARGETS_IN_DRUGBANK': 'first',
    'PERCENTAGE_OF_TARGETS_HIT': 'max',
    'ACTION': lambda x: ', '.join(x),
    'SPECIFIC_FUNCTION': lambda x: ', '.join(x),
    'drug_hypergeom_fdr': 'max',
    'drug_empirical_fdr': 'max',
    'Target_Description': lambda x: ', '.join(x)
}).reset_index()

/var/folders/5p/swntgnbj3fbfxkx02kt3fq980000gn/T/ipykernel_1907/4020300996.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  hpv_positive_final_direct['Target_Description'] = 'Direct'


In [46]:
hpv_positive_final_results.sort_values(by = ['drug_empirical_fdr', 'PERCENTAGE_OF_TARGETS_HIT'], ascending = [ True, False ]).head(50)

,DRUG,GENE_TARGET,CONNECTED_TO (risk gene),NUM_DIRECT_TARGETS_HIT,Number of risk or immediate neighbor genes targeted,TOTAL_TARGETS_IN_DRUGBANK,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_fdr,drug_empirical_fdr,Target_Description
32,ci-1040,"AKT1,LCK, RPS6KB1, AKT1, MAPK1,AKT1,RPS6KB1,CH...","SOX2,PIK3CA,ADIPOQ,EIF4A2,RFC4,GNB4,MRPL47",NA,10.0,NA,66.666667,inhibitor,14-3-3 protein binding,1.910923e-03,0.002095,Indirect
18,artenimol,"GAPDH, GAPDH,NPM1,ALB,ANXA2, ALB,CCT3, HSPA8, ...","SOX2,PIK3CA,ADIPOQ,AHSG,DNAJB11,EIF4A2,FXR1,HR...",NA,36.0,NA,34.615385,ligand,aspartic-type endopeptidase inhibitor activity...,4.922768e-05,0.002095,Indirect
25,brigatinib,"EGFR,IGF1R, MET, ALK, ERBB4, INSR, EGFR, ABL1,...","SOX2,PIK3CA,IGF2BP2,SH3KBP1,VEGFD, PIK3CA",NA,9.0,NA,100.000000,"inhibitor, inhibitor, binding","actin filament binding, ATP binding, amyloid-b...",1.532820e-05,0.002309,Indirect
45,erdafitinib,"FGFR1, FGFR4, FGFR3, KDR, PDGFRB, KIT, CSF1R, ...","PIK3CA,IL1RAP,ANOS1,RPS6KA3,VEGFD, PIK3CA",NA,10.0,NA,100.000000,"inhibitor, substrate, inhibitor, substrate","ATP binding, ATP binding",2.923712e-06,0.002309,Indirect
46,famitinib,"KDR, PDGFRB, FLT1, KIT, PDGFRA, FLT4, FLT3,KIT...","PIK3CA,IL1RAP,VEGFD, PIK3CA",NA,7.0,NA,100.000000,"inhibitor, inhibitor","ATP binding, ATP binding",4.376727e-04,0.002309,Indirect
65,lenvatinib,"FGFR1, FGFR4, FGFR3, KDR, FLT1, KIT, RET, PDGF...","PIK3CA,IL1RAP,ANOS1,RPS6KA3,VEGFD, PIK3CA",NA,10.0,NA,100.000000,"inhibitor, inhibitor","ATP binding, ATP binding",2.923712e-06,0.002309,Indirect
70,lucitanib,"FGFR1, FGFR3, KDR, PDGFRB, FLT1, PDGFRA, FGFR2...","PIK3CA,ANOS1,RPS6KA3,VEGFD, PIK3CA",NA,8.0,NA,100.000000,"inhibitor, inhibitor","ATP binding, ATP binding",8.423169e-05,0.002309,Indirect
77,nintedanib,"FGFR1, FGFR3, SRC, KDR, PDGFRB, LCK, FLT1, PDG...","PIK3CA,KNG1,ANOS1,RPS6KA3,SH3KBP1,VEGFD, PIK3C...",NA,12.0,NA,100.000000,"inhibitor, inhibitor, inhibitor","ATP binding, ATP binding, ATP binding",1.946417e-07,0.002309,Indirect
86,pd-166326,"EGFR,FGFR1, SRC, PDGFRB, LCK, EGFR, KIT, ABL1,...","SOX2,PIK3CA,KNG1,GNB4,IGF2BP2,IL1RAP,ANOS1,RPS...",NA,9.0,NA,100.000000,"inhibitor, inhibitor, inhibitor","actin filament binding, ATP binding, actin fil...",1.532820e-05,0.002309,Indirect
112,sorafenib,"EGFR,RAF1, FGFR1, KDR, PDGFRB, EGFR, FLT1, BRA...","SOX2,PIK3CA,IGF2BP2,IL1RAP,ANOS1,CNKSR2,RPS6KA...",NA,11.0,NA,100.000000,"inhibitor, inhibitor, antagonist","actin filament binding, ATP binding, actin fil...",9.806338e-07,0.002309,Indirect


In [47]:
len(set(hpv_positive_final_results['DRUG']))

127

## HPV-

#### Genes

In [48]:
### combine hpv negative somatic genes and cnv genes
hpv_negative_som_genes['MUT_TYPE'] = 'SOMATIC'
hpv_negative_som_genes['gene_name'] = hpv_negative_som_genes['Gene']
hpv_negative_som_genes['q_value']= hpv_negative_som_genes['Adjusted_P_Value']
hpv_negative_som_genes['empirical_q_value'] = hpv_negative_som_genes['Adjusted_Empirical_P_Value']
hpv_negative_combined_genes = pd.concat([hpv_negative_genes, hpv_negative_som_genes], axis=0)
### aggregate by GENE to get unique genes with both mutation types
hpv_negative_combined_genes['gene_name'] = hpv_negative_combined_genes['gene_name'].str.upper()
extracted_target_df_combined['GENE'] = extracted_target_df_combined['GENE'].str.upper()
hpv_negative_combined_genes = hpv_negative_combined_genes.groupby('gene_name').agg({
    'MUT_TYPE': lambda x: ', '.join(x),
    'q_value': lambda x: ', '.join(x.astype(str)) if len(x) > 1 else x.iloc[0].astype(str),
    'empirical_q_value': lambda x: ', '.join(x.astype(str)) if len(x) > 1 else x.iloc[0].astype(str)
}).reset_index()
# hpv_negative_combined_genes.sort_values(by='q_value')

In [49]:
hpv_negative_combined_genes

,gene_name,MUT_TYPE,q_value,empirical_q_value
0,ABCF3,AMPLIFICATION,1.4051416817717394e-199,0.0043585777302941
1,ACTL6A,AMPLIFICATION,4.6046768654042024e-207,0.0043585777302941
2,ADCY2,SOMATIC,0.0003377698987821,0.005960073448722
3,ADCY8,SOMATIC,0.0021845458816456,0.005960073448722
4,ADGRB3,SOMATIC,8.330155899454921e-05,0.005960073448722
...,...,...,...,...
228,ZNF676,SOMATIC,0.0239614999999452,0.005960073448722
229,ZNF804A,SOMATIC,0.003536214310732,0.005960073448722
230,ZNF804B,SOMATIC,5.057316865018672e-06,0.005960073448722
231,ZNF835,SOMATIC,1.0196847272849249e-05,0.005960073448722


In [50]:
### merge genes with number of articles, pubmed id from literature data
hpv_negative_gene_results_with_lit = pd.merge(hpv_negative_combined_genes, extracted_target_df_combined, how = 'left', left_on='gene_name', right_on='GENE')
hpv_negative_gene_results_with_lit.drop(columns = ['INDEX'], inplace = True)

In [51]:
hpv_negative_gene_results_with_lit = hpv_negative_gene_results_with_lit[hpv_negative_gene_results_with_lit['NUMBER_OF_ARTICLES']>0]

In [52]:
hpv_negative_gene_results_with_lit.sort_values(by = ['empirical_q_value', 'q_value'], ascending=[True, True], inplace=True)

In [53]:
hpv_negative_gene_results_with_lit.drop(columns=['INDEX', 'GENE'], errors='ignore', inplace=True)

In [54]:
hpv_negative_gene_results_with_lit.to_csv('Results/HPV negative gene results.csv')

In [55]:
hpv_negative_gene_results_with_lit

,gene_name,MUT_TYPE,q_value,empirical_q_value,PMID,NUMBER_OF_ARTICLES
51,EIF4G1,AMPLIFICATION,1.4051416817717394e-199,0.0043585777302941,14676830,1.0
202,SOX2,AMPLIFICATION,1.960361075687212e-205,0.0043585777302941,15942670,1.0
45,DVL3,AMPLIFICATION,2.0584024908359247e-200,0.0043585777302941,"14676830, 16865240",2.0
177,PRKCI,AMPLIFICATION,3.026251441733312e-201,0.0043585777302941,17990328,1.0
165,PDCD10,AMPLIFICATION,3.0744600814299213e-196,0.0043585777302941,17409414,1.0
26,CLDN1,AMPLIFICATION,4.2419726156021995e-197,0.0043585777302941,"15170668, 17091452",2.0
12,BCL6,AMPLIFICATION,5.884151323631562e-198,0.0043585777302941,"11224600, 11420458, 14685876, 17429099",4.0
125,MYNN,AMPLIFICATION,8.446185897603275e-199,0.0043585777302941,17409414,1.0
183,RFC4,AMPLIFICATION,8.446185897603275e-199,0.0043585777302941,16467079,1.0
170,PIK3CA,"AMPLIFICATION, SOMATIC","1.5505691778987169e-208, 8.841194236005348e-40","0.0043585777302941, 0.005960073448722","11358835, 11836556, 11959846, 14581353, 155436...",17.0


In [56]:
len(set(hpv_negative_gene_results_with_lit['gene_name']))

29

In [57]:
### export top genes to ouput final tables
hpv_negative_gene_results_with_lit.to_csv('Results/Final Results/HPV Negative validated genes.csv')

#### Direct

In [58]:
### combine all hpv negative direct drug candidates
hpv_negative_final_direct = pd.concat([hpv_negative_direct_drug_candidates, hpv_negative_som_direct_drug_candidates])
### group by drug and comma seperate genes and mutation type
### columns: DRUG	GENE_TARGET	NUM_DIRECT_TARGETS_HIT	TOTAL_TARGETS_IN_DRUGBANK	PERCENTAGE_OF_TARGETS_HIT	GENE_GISTIC	GENE_normalized_gistic_score	
# ACTION	SPECIFIC_FUNCTION	drug_hypergeom_p_value      

hpv_negative_final_direct = hpv_negative_final_direct.groupby('DRUG').agg({'GENE_TARGET': lambda x: ', '.join(x),
                                               'MUT_TYPE': lambda x: ', '.join(x.unique()), ### unique mutation types per drug
                                                  'NUM_DIRECT_TARGETS_HIT': 'first',
                                                    'TOTAL_TARGETS_IN_DRUGBANK': 'first',
                                                    'PERCENTAGE_OF_TARGETS_HIT': 'first',
                                                    'ACTION': 'first',
                                                    'SPECIFIC_FUNCTION': 'first',
                                                    'drug_hypergeom_p_value': 'first',
                                                    'drug_hypergeom_fdr': 'first',
                                                    'drug_empirical_p_value': 'first',
                                                    'drug_empirical_fdr': 'first',
                                                    }).reset_index()

In [59]:
extracted_target_df_combined[extracted_target_df_combined['GENE'] == 'EPHA2']

,GENE,PMID,INDEX,NUMBER_OF_ARTICLES
2074,EPHA2,"12494475, 16309192, 18030354, 18425361, 18485799","13831, 40865, 57416, 61136, 61446",5


In [60]:
### validate hpv negative direct drug candidates with literature data
hpv_negative_final_direct['PMID'] = ''
hpv_negative_final_direct['NUMBER_OF_ARTICLES'] = 0
hpv_negative_final_direct['LITERATURE_GENE_TARGETS'] = ''
for index, row in hpv_negative_final_direct.iterrows():
    gene_targets = row['GENE_TARGET'].split(', ')
    gene_targets = [gene.strip() for gene in gene_targets]
    gene_targets = list(set(gene_targets))
    pmids_set = set()
    literature_gene_targets = set()
    number_of_articles = 0
    for gene in gene_targets:
        matched_rows = extracted_target_df_combined[extracted_target_df_combined['GENE'] == gene]
        for _, matched_row in matched_rows.iterrows():
            pmids = matched_row['PMID'].split(', ')
            pmids_set.update(pmids)
            number_of_articles += matched_row['NUMBER_OF_ARTICLES']
            literature_gene_targets.add(matched_row['GENE'])
    
    hpv_negative_final_direct.at[index, 'LITERATURE_GENE_TARGETS'] = ', '.join(literature_gene_targets)
    hpv_negative_final_direct.at[index, 'PMID'] = ', '.join(pmids_set)
    hpv_negative_final_direct.at[index, 'NUMBER_OF_ARTICLES'] = number_of_articles

    

In [61]:
hpv_negative_final_direct[hpv_negative_final_direct['NUMBER_OF_ARTICLES'] > 0].sort_values(by='NUMBER_OF_ARTICLES', ascending=False)

,DRUG,GENE_TARGET,MUT_TYPE,NUM_DIRECT_TARGETS_HIT,TOTAL_TARGETS_IN_DRUGBANK,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_p_value,drug_hypergeom_fdr,drug_empirical_p_value,drug_empirical_fdr,PMID,NUMBER_OF_ARTICLES,LITERATURE_GENE_TARGETS
0,Acetylsalicylic acid,TP53,SOMATIC,1.0,20.0,5.000000,inducer,14-3-3 protein binding,9.445065e-07,4.726206e-04,0.00001,0.003603,"17258495, 15810068, 12673679, 15965904, 168478...",29,TP53
5,Caffeine,PIK3CA,SOMATIC,1.0,15.0,6.666667,inhibitor,1-phosphatidylinositol-3-kinase activity,1.445445e-04,2.503677e-02,0.00013,0.023418,"11959846, 15700036, 16807070, 17990317, 166763...",17,PIK3CA
20,Wortmannin,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1.0,5.0,20.000000,None,1-phosphatidylinositol-3-kinase activity,4.040415e-04,4.043557e-02,0.00030,0.043582,"11959846, 15700036, 16807070, 17990317, 166763...",17,PIK3CA
21,XL765,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1.0,5.0,20.000000,None,1-phosphatidylinositol-3-kinase activity,4.040415e-04,4.043557e-02,0.00036,0.047036,"11959846, 15700036, 16807070, 17990317, 166763...",17,PIK3CA
12,Fostamatinib,"MAP3K13, PRKCI, TNIK, TGFBR2, EPHA7, EPHA2, EPHA5","AMPLIFICATION, SOMATIC",3.0,300.0,1.000000,inhibitor,ATP binding,6.055938e-10,7.792261e-07,0.00001,0.002502,"18030354, 18425361, 17990328, 12494475, 163091...",6,"EPHA2, PRKCI"
8,Dasatinib,"EPHA2, EPHA5",SOMATIC,2.0,23.0,8.695652,antagonist,ATP binding,5.520155e-08,5.524449e-05,0.00001,0.003603,"18030354, 18425361, 12494475, 16309192, 18485799",5,EPHA2
17,Regorafenib,EPHA2,SOMATIC,1.0,18.0,5.555556,inhibitor,ATP binding,9.641629e-10,4.342107e-06,0.00001,0.003603,"18030354, 18425361, 12494475, 16309192, 18485799",5,EPHA2
2,Bisindolylmaleimide I,PRKCI,AMPLIFICATION,1.0,19.0,5.263158,None,ATP binding,1.007861e-07,6.051867e-05,0.00001,0.002502,17990328,1,PRKCI
4,Bryostatin 1,CASP8,SOMATIC,1.0,9.0,11.111111,inhibitor,cysteine-type endopeptidase activity,2.354554e-05,6.547736e-03,0.00004,0.011622,16857411,1,CASP8
13,Heparin,SELP,SOMATIC,1.0,12.0,8.333333,inhibitor,calcium ion binding,7.836423e-05,1.501759e-02,0.00008,0.016757,16135921,1,SELP


In [62]:
### ensure that only drugs with NUMBER_OF_ARTICLES > 0 are saved, so that they have literature support
hpv_negative_final_direct= hpv_negative_final_direct[hpv_negative_final_direct['NUMBER_OF_ARTICLES'] > 0]
### save results
hpv_negative_final_direct.to_csv('Results/HPV Negative direct results.csv')

In [63]:
hpv_negative_final_direct

,DRUG,GENE_TARGET,MUT_TYPE,NUM_DIRECT_TARGETS_HIT,TOTAL_TARGETS_IN_DRUGBANK,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_p_value,drug_hypergeom_fdr,drug_empirical_p_value,drug_empirical_fdr,PMID,NUMBER_OF_ARTICLES,LITERATURE_GENE_TARGETS
0,Acetylsalicylic acid,TP53,SOMATIC,1.0,20.0,5.000000,inducer,14-3-3 protein binding,9.445065e-07,4.726206e-04,0.00001,0.003603,"17258495, 15810068, 12673679, 15965904, 168478...",29,TP53
2,Bisindolylmaleimide I,PRKCI,AMPLIFICATION,1.0,19.0,5.263158,None,ATP binding,1.007861e-07,6.051867e-05,0.00001,0.002502,17990328,1,PRKCI
4,Bryostatin 1,CASP8,SOMATIC,1.0,9.0,11.111111,inhibitor,cysteine-type endopeptidase activity,2.354554e-05,6.547736e-03,0.00004,0.011622,16857411,1,CASP8
5,Caffeine,PIK3CA,SOMATIC,1.0,15.0,6.666667,inhibitor,1-phosphatidylinositol-3-kinase activity,1.445445e-04,2.503677e-02,0.00013,0.023418,"11959846, 15700036, 16807070, 17990317, 166763...",17,PIK3CA
8,Dasatinib,"EPHA2, EPHA5",SOMATIC,2.0,23.0,8.695652,antagonist,ATP binding,5.520155e-08,5.524449e-05,0.00001,0.003603,"18030354, 18425361, 12494475, 16309192, 18485799",5,EPHA2
12,Fostamatinib,"MAP3K13, PRKCI, TNIK, TGFBR2, EPHA7, EPHA2, EPHA5","AMPLIFICATION, SOMATIC",3.0,300.0,1.000000,inhibitor,ATP binding,6.055938e-10,7.792261e-07,0.00001,0.002502,"18030354, 18425361, 17990328, 12494475, 163091...",6,"EPHA2, PRKCI"
13,Heparin,SELP,SOMATIC,1.0,12.0,8.333333,inhibitor,calcium ion binding,7.836423e-05,1.501759e-02,0.00008,0.016757,16135921,1,SELP
17,Regorafenib,EPHA2,SOMATIC,1.0,18.0,5.555556,inhibitor,ATP binding,9.641629e-10,4.342107e-06,0.00001,0.003603,"18030354, 18425361, 12494475, 16309192, 18485799",5,EPHA2
20,Wortmannin,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1.0,5.0,20.000000,None,1-phosphatidylinositol-3-kinase activity,4.040415e-04,4.043557e-02,0.00030,0.043582,"11959846, 15700036, 16807070, 17990317, 166763...",17,PIK3CA
21,XL765,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1.0,5.0,20.000000,None,1-phosphatidylinositol-3-kinase activity,4.040415e-04,4.043557e-02,0.00036,0.047036,"11959846, 15700036, 16807070, 17990317, 166763...",17,PIK3CA


#### Indirect

In [64]:
hpv_negative_indirect_drug_candidates[hpv_negative_indirect_drug_candidates['DRUG'].str.lower()== 'artenimol']

,DRUG,GENE_TARGET,CONNECTED_TO (risk gene),Number of risk or immediate neighbor genes targeted,total_genes_targeted_in_drugbank,PERCENTAGE_OF_TARGETS_HIT,Number of indirect genes connected to this risk gene,GENE_GISTIC,GENE_normalized_gistic_score,GENE_frequency_percentage,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_fdr,drug_empirical_fdr,MUT_TYPE
9,artenimol,"HNRNPK, DDX5,NPM1,ACTG1,CYCS,RPS9, RPS13, RPS1...","FXR1,PIK3CA,ACTL6A,MFN1,MRPL47,NDUFB5,SOX2,TNF...",40,104,38.461538,15,"0.3209027981846753,0.3202613210014027,0.318473...","0.6499866156631651,0.6298428966518568,0.573712...","36.02110022607385,36.02110022607385,35.8703843...",ligand,"cadherin binding, ATP binding",0.002629,0.002502,"DELETION,AMPLIFICATION"


In [65]:
### combine all hpv negative indirect drug candidates
hpv_negative_final_indirect = pd.concat([hpv_negative_indirect_drug_candidates, hpv_negative_som_indirect_drug_candidates])
hpv_negative_final_indirect['ACTION'] = hpv_negative_final_indirect['ACTION'].fillna('UNKNOWN')
hpv_negative_final_indirect['SPECIFIC_FUNCTION'] = hpv_negative_final_indirect['SPECIFIC_FUNCTION'].fillna('UNKNOWN')
hpv_negative_final_indirect['drug_hypergeom_fdr'] = hpv_negative_final_indirect['drug_hypergeom_fdr'].fillna('UNKNOWN')
hpv_negative_final_indirect['drug_empirical_fdr'] = hpv_negative_final_indirect['drug_empirical_fdr'].fillna('UNKNOWN')
hpv_negative_final_indirect['MUT_TYPE'] = hpv_negative_final_indirect['MUT_TYPE'].fillna('UNKNOWN')

### group by drug and comma seperate genes and mutation type
### columns: DRUG	CONNECTED_TO (risk gene)
# Number of risk or immediate neighbor genes
# targeted	total_genes_targeted_in_drugbank  
# PERCENTAGE_OF_TARGETS_HIT
# Number of indirect genes connected to this risk gene  
# GENE_TARGET	
# GENE_Cohort_Frequency   
# GENE_Normalized_Count	
# GENE_Normalized_Cohort_Frequency

hpv_negative_final_indirect = hpv_negative_final_indirect.groupby (['DRUG']).agg({
    'GENE_TARGET': lambda x: ', '.join(x),
    'CONNECTED_TO (risk gene)': lambda x: ', '.join(x),
    'Number of risk or immediate neighbor genes targeted': 'first',
    'total_genes_targeted_in_drugbank': 'first',
    'PERCENTAGE_OF_TARGETS_HIT': 'first',
    'ACTION': lambda x: ', '.join(x),
    'SPECIFIC_FUNCTION': lambda x: ', '.join(x),
    'drug_hypergeom_fdr': 'max',
    'drug_empirical_fdr': 'max',
    'MUT_TYPE': lambda x: ', '.join(x)
}).reset_index()

In [66]:
'artenimol' in hpv_negative_final_indirect['DRUG'].str.lower().values

True

In [67]:
hpv_negative_final_indirect.sort_values(by = 'PERCENTAGE_OF_TARGETS_HIT', ascending = True).head(20)

,DRUG,GENE_TARGET,CONNECTED_TO (risk gene),Number of risk or immediate neighbor genes targeted,total_genes_targeted_in_drugbank,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_fdr,drug_empirical_fdr,MUT_TYPE
49,fostamatinib,"CAMK2A,PRKACA, PRKG2, PRKACB,PRKCD, ROS1, AXL,...","FXR1,KCNMB3,PIK3CA,KCNMB2,GNB4,MFN1,MRPL47,SOX...",105,299,35.117057,"inhibitor, inhibitor, inhibitor, inhibitor, in...","ATP binding, ATP binding, actin binding, [hydr...",7.792261e-07,0.003603,"AMPLIFICATION, SOMATIC, SOMATIC, SOMATIC, SOMA..."
13,artenimol,"HNRNPK, DDX5,NPM1,ACTG1,CYCS,RPS9, RPS13, RPS1...","FXR1,PIK3CA,ACTL6A,MFN1,MRPL47,NDUFB5,SOX2,TNF...",40,104,38.461538,ligand,"cadherin binding, ATP binding",2.629444e-03,0.002502,"DELETION,AMPLIFICATION"
72,nadh,"NDUFA8, NDUFS5, NDUFS2, NDUFA2, NDUFB10, NDUFV...",NDUFB5,58,144,40.277778,binder,"NADH dehydrogenase (ubiquinone) activity, 4 ir...",1.996032e-05,0.002502,AMPLIFICATION
8,alvocidib,"CDK1, CDK2, CDK5, EGFR, CDK4, CDK6,CDK2, CDK4,...","CDKN2A,CDKN2B, TP53, CDKN2A, CASP8",6,12,50.000000,"inhibitor, inhibitor, inhibitor, inhibitor","ATP binding, acetylcholine receptor activator ...",1.501759e-02,0.023418,"DELETION, SOMATIC, SOMATIC, SOMATIC"
25,calcium citrate,"S100B, S100A6, TPT1, CALM3, CANX, CALR, CANX, ...","TP53, HRAS, HLA-B, HLA-A, RHOA, RAC1, TGFBR2, ...",17,33,51.515152,"ligand, agonist, ligand, ligand, agonist, agon...","calcium ion binding, adenylate cyclase activat...",1.501759e-02,0.016757,"SOMATIC, SOMATIC, SOMATIC, SOMATIC, SOMATIC, S..."
26,calcium phosphate,"S100B, S100A6, TPT1, CALM3, CANX, CALR, CANX, ...","TP53, HRAS, HLA-B, HLA-A, RHOA, RAC1, TGFBR2, ...",17,33,51.515152,"ligand, agonist, ligand, ligand, agonist, agon...","calcium ion binding, adenylate cyclase activat...",1.501759e-02,0.016757,"SOMATIC, SOMATIC, SOMATIC, SOMATIC, SOMATIC, S..."
33,clozapine,"HTR1D, HTR1A, HTR7, DRD4, HTR1B, HTR1E, DRD2, ...","GNB4,SST,HTR3C,HTR3D,HTR3E,CLCN2,KNG1",17,33,51.515152,"antagonist, partial agonist","G protein-coupled serotonin receptor activity,...",1.507498e-02,0.010596,AMPLIFICATION
124,ziprasidone,"HTR1D, HTR1A, HTR7, DRD4, HTR1B, HTR1E, DRD2, ...","GNB4,SST,HTR3C,HTR3D,HTR3E,KNG1",14,27,51.851852,"antagonist, agonist","G protein-coupled serotonin receptor activity,...",4.043557e-02,0.045749,AMPLIFICATION
91,purvalanol,"CDK1, CDK2, CDK5, MAPK1, CDK4,CDK2, CDK4","CDKN2A,CDKN2B",5,9,55.555556,"binder, inhibitor","ATP binding, acetylcholine receptor activator ...",1.597552e-03,0.006005,DELETION
11,aripiprazole lauroxil,"HTR1D, HTR1A, HTR7, DRD4, HTR1B, HTR1E, DRD2, ...","GNB4,SST",14,25,56.000000,partial agonist,"G protein-coupled serotonin receptor activity,...",1.891827e-02,0.013595,AMPLIFICATION


In [68]:
len(hpv_negative_final_indirect[hpv_negative_final_indirect['DRUG']=='fostamatinib']['GENE_TARGET'].iloc[0].split(','))

424

In [69]:
### extract the unique number of GENE_TARGETs from , seperated list in each of the GENE_TARGET column
def extract_unique_gene_targets(df, column_name):
    return len(set([gene.strip() for sublist in df[column_name].dropna().str.split(',') for gene in sublist]))

hpv_negative_indirect_genes = extract_unique_gene_targets(hpv_negative_final_indirect, 'GENE_TARGET')
print(hpv_negative_indirect_genes)

444


In [70]:
### add in literature validation columns
hpv_negative_final_indirect['PMID'] = ''
hpv_negative_final_indirect['NUMBER_OF_ARTICLES'] = 0
hpv_negative_final_indirect['LITERATURE_GENE_TARGETS'] = ''
for index, row in hpv_negative_final_indirect.iterrows():
    gene_targets = row['GENE_TARGET'].split(',')
    gene_targets = [gene.strip() for gene in gene_targets]
    pmids_set = set()
    literature_gene_targets = set()
    number_of_articles = 0
    for gene in gene_targets:
        matched_rows = extracted_target_df_combined[extracted_target_df_combined['GENE'] == gene]
        for _, matched_row in matched_rows.iterrows():
            pmids = matched_row['PMID'].split(',')
            pmids = [pmid.strip() for pmid in pmids]
            pmids_set.update(pmids)
            number_of_articles += matched_row['NUMBER_OF_ARTICLES']
            literature_gene_targets.add(matched_row['GENE'])
    
    hpv_negative_final_indirect.at[index, 'LITERATURE_GENE_TARGETS'] = ', '.join(literature_gene_targets)
    hpv_negative_final_indirect.at[index, 'PMID'] = ', '.join(pmids_set)
    hpv_negative_final_indirect.at[index, 'NUMBER_OF_ARTICLES'] = number_of_articles


### validate risk genes
hpv_negative_final_indirect = hpv_negative_final_indirect[hpv_negative_final_indirect['NUMBER_OF_ARTICLES'] > 0]
hpv_negative_final_indirect['RISK_GENE_PMID'] = ''
hpv_negative_final_indirect['RISK_GENE_NUMBER_OF_ARTICLES'] = 0
hpv_negative_final_indirect['RISK_GENE_LITERATURE_GENE_TARGETS'] = ''
for index, row in hpv_negative_final_indirect.iterrows():
    risk_genes = row['CONNECTED_TO (risk gene)'].split(',')
    risk_genes = [gene.strip() for gene in risk_genes]
    pmids_set = set()
    literature_gene_targets = set()
    number_of_articles = 0
    for gene in risk_genes:
        matched_rows = extracted_target_df_combined[extracted_target_df_combined['GENE'] == gene]
        for _, matched_row in matched_rows.iterrows():
            pmids = matched_row['PMID'].split(',')
            pmids = [pmid.strip() for pmid in pmids]
            pmids_set.update(pmids)
            number_of_articles += matched_row['NUMBER_OF_ARTICLES']
            literature_gene_targets.add(matched_row['GENE'])
    
    hpv_negative_final_indirect.at[index, 'RISK_GENE_LITERATURE_GENE_TARGETS'] = ', '.join(literature_gene_targets)
    hpv_negative_final_indirect.at[index, 'RISK_GENE_PMID'] = ', '.join(pmids_set)
    hpv_negative_final_indirect.at[index, 'RISK_GENE_NUMBER_OF_ARTICLES'] = number_of_articles

### make sure only drugs with literature support for both drug targets and risk genes are saved
hpv_negative_final_indirect = hpv_negative_final_indirect[hpv_negative_final_indirect['NUMBER_OF_ARTICLES'] > 0]
hpv_negative_final_indirect = hpv_negative_final_indirect[hpv_negative_final_indirect['RISK_GENE_NUMBER_OF_ARTICLES'] > 0]

hpv_negative_final_indirect.to_csv('Results/HPV Negative indirect results.csv', index=False)



In [71]:
hpv_negative_final_indirect.sort_values(by = ['drug_empirical_fdr', 'PERCENTAGE_OF_TARGETS_HIT'], ascending=[True, False]).head(50)[['DRUG', 'drug_empirical_fdr','LITERATURE_GENE_TARGETS', 'RISK_GENE_PMID','RISK_GENE_LITERATURE_GENE_TARGETS','RISK_GENE_NUMBER_OF_ARTICLES','MUT_TYPE']]

,DRUG,drug_empirical_fdr,LITERATURE_GENE_TARGETS,RISK_GENE_PMID,RISK_GENE_LITERATURE_GENE_TARGETS,RISK_GENE_NUMBER_OF_ARTICLES,MUT_TYPE
92,quercetin,0.002502,"ESR1, SHBG, HCK, STAT3","17990317, 11420458, 15543611, 16676365, 145813...","PIK3CA, SOX2, BCL6",22,AMPLIFICATION
13,artenimol,0.002502,"RPS13, EEF1A1, ANXA2, HSPA8, RPS6, RPL14, GAPDH","14676830, 17079134, 17990317, 15543611, 146547...","PIK3CA, CDKN2A, EIF4G1, SOX2",35,"DELETION,AMPLIFICATION"
22,brigatinib,0.003603,"ERBB4, ERBB2, MET, IGF1R, EGFR, ALK","17079134, 17990317, 17258495, 16219138, 175925...","RAC1, KEAP1, CDKN2A, CASP8, TP53, HLA-A, RHOA,...",112,"DELETION,AMPLIFICATION, SOMATIC, SOMATIC, SOMA..."
40,erdafitinib,0.003603,"RET, FGFR3, PDGFRA, KIT, FGFR1, FGFR2, FGFR4","17079134, 17990317, 17258495, 16219138, 125390...","CDKN2A, NOTCH1, TP53, RHOA, PIK3CA, HRAS",85,"AMPLIFICATION, SOMATIC, SOMATIC, SOMATIC, SOMA..."
60,lenvatinib,0.003603,"RET, FGFR3, PDGFRA, KIT, FGFR1, FGFR2, FGFR4","17079134, 17990317, 17258495, 16219138, 125390...","CDKN2A, NOTCH1, TP53, RHOA, PIK3CA, HRAS",85,"AMPLIFICATION, SOMATIC, SOMATIC, SOMATIC, SOMA..."
75,nintedanib,0.003603,"FGFR3, PDGFRA, LYN, SRC, FGFR1, FGFR2","17079134, 17990317, 17258495, 16219138, 175925...","RAC1, CDKN2A, CASP8, TP53, RHOA, NOTCH1, DCC, ...",96,"AMPLIFICATION, SOMATIC, SOMATIC, SOMATIC, SOMA..."
81,pd-166326,0.003603,"PDGFRA, KIT, SRC, FGFR1, EGFR","17079134, 17258495, 16637057, 16676365, 145813...","RAC1, KEAP1, CDKN2A, CASP8, TP53, RHOA, NOTCH1...",114,"DELETION,AMPLIFICATION, SOMATIC, SOMATIC, SOMA..."
101,seliciclib,0.003603,"CDK2, CDK1","17079134, 17258495, 15810068, 12673679, 168574...","CDKN2A, CASP8, TP53",46,"SOMATIC, SOMATIC, SOMATIC"
107,sorafenib,0.003603,"RET, KIT, BRAF, FGFR1, EGFR","17079134, 17990317, 17258495, 16219138, 125390...","KEAP1, CDKN2A, CASP8, TP53, RHOA, NOTCH1, PIK3...",93,"AMPLIFICATION, SOMATIC, SOMATIC, SOMATIC, SOMA..."
110,sunitinib,0.003603,"KIT, MET, PDGFRA","17079134, 17990317, 17258495, 17592548, 158100...","RAC1, CDKN2A, TP53, RHOA, PIK3CA, HRAS",85,"AMPLIFICATION, SOMATIC, SOMATIC, SOMATIC, SOMA..."


#### overall

In [72]:
hpv_negative_final_direct

,DRUG,GENE_TARGET,MUT_TYPE,NUM_DIRECT_TARGETS_HIT,TOTAL_TARGETS_IN_DRUGBANK,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_p_value,drug_hypergeom_fdr,drug_empirical_p_value,drug_empirical_fdr,PMID,NUMBER_OF_ARTICLES,LITERATURE_GENE_TARGETS
0,Acetylsalicylic acid,TP53,SOMATIC,1.0,20.0,5.000000,inducer,14-3-3 protein binding,9.445065e-07,4.726206e-04,0.00001,0.003603,"17258495, 15810068, 12673679, 15965904, 168478...",29,TP53
2,Bisindolylmaleimide I,PRKCI,AMPLIFICATION,1.0,19.0,5.263158,None,ATP binding,1.007861e-07,6.051867e-05,0.00001,0.002502,17990328,1,PRKCI
4,Bryostatin 1,CASP8,SOMATIC,1.0,9.0,11.111111,inhibitor,cysteine-type endopeptidase activity,2.354554e-05,6.547736e-03,0.00004,0.011622,16857411,1,CASP8
5,Caffeine,PIK3CA,SOMATIC,1.0,15.0,6.666667,inhibitor,1-phosphatidylinositol-3-kinase activity,1.445445e-04,2.503677e-02,0.00013,0.023418,"11959846, 15700036, 16807070, 17990317, 166763...",17,PIK3CA
8,Dasatinib,"EPHA2, EPHA5",SOMATIC,2.0,23.0,8.695652,antagonist,ATP binding,5.520155e-08,5.524449e-05,0.00001,0.003603,"18030354, 18425361, 12494475, 16309192, 18485799",5,EPHA2
12,Fostamatinib,"MAP3K13, PRKCI, TNIK, TGFBR2, EPHA7, EPHA2, EPHA5","AMPLIFICATION, SOMATIC",3.0,300.0,1.000000,inhibitor,ATP binding,6.055938e-10,7.792261e-07,0.00001,0.002502,"18030354, 18425361, 17990328, 12494475, 163091...",6,"EPHA2, PRKCI"
13,Heparin,SELP,SOMATIC,1.0,12.0,8.333333,inhibitor,calcium ion binding,7.836423e-05,1.501759e-02,0.00008,0.016757,16135921,1,SELP
17,Regorafenib,EPHA2,SOMATIC,1.0,18.0,5.555556,inhibitor,ATP binding,9.641629e-10,4.342107e-06,0.00001,0.003603,"18030354, 18425361, 12494475, 16309192, 18485799",5,EPHA2
20,Wortmannin,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1.0,5.0,20.000000,None,1-phosphatidylinositol-3-kinase activity,4.040415e-04,4.043557e-02,0.00030,0.043582,"11959846, 15700036, 16807070, 17990317, 166763...",17,PIK3CA
21,XL765,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1.0,5.0,20.000000,None,1-phosphatidylinositol-3-kinase activity,4.040415e-04,4.043557e-02,0.00036,0.047036,"11959846, 15700036, 16807070, 17990317, 166763...",17,PIK3CA


In [73]:
### combined direct and indirect final results
### final columns: DRUG   GENE_TARGET CONNECTED_TO (risk gene)	NUM_DIRECT_TARGETS_HIT  
# Number of risk or immediate neighbor genes targeted	TOTAL_TARGETS_IN_DRUGBANK	PERCENTAGE_OF_TARGETS_HIT	
# ACTION	SPECIFIC_FUNCTION	drug_hypergeom_fdr	drug_empirical_fdr	
### column for Target description: direct, or indirect
hpv_negative_final_direct['Target_Description'] = 'Direct'
hpv_negative_final_indirect['Target_Description'] = 'Indirect'
hpv_negative_final_results = pd.concat([hpv_negative_final_direct, hpv_negative_final_indirect], ignore_index=True)
### replace any null with 'NA' in the hwole dataframe
hpv_negative_final_results = hpv_negative_final_results.fillna('NA')
hpv_negative_final_results = hpv_negative_final_results.groupby('DRUG').agg({
    'GENE_TARGET': lambda x: ', '.join(x),
    'CONNECTED_TO (risk gene)': lambda x: ', '.join(x),
    'NUM_DIRECT_TARGETS_HIT': 'first',
    'Number of risk or immediate neighbor genes targeted': 'first',
    'TOTAL_TARGETS_IN_DRUGBANK': 'first',
    'PERCENTAGE_OF_TARGETS_HIT': 'max',
    'ACTION': lambda x: ', '.join(x),
    'SPECIFIC_FUNCTION': lambda x: ', '.join(x),
    'drug_hypergeom_fdr': 'max',
    'drug_empirical_fdr': 'max',
    'Target_Description': lambda x: ', '.join(x)
}).reset_index()

In [74]:
len(set(hpv_negative_final_results['DRUG']))

80

In [75]:
hpv_positive_final_direct

,DRUG,GENE_TARGET,MUT_TYPE,NUM_DIRECT_TARGETS_HIT,TOTAL_TARGETS_IN_DRUGBANK,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_p_value,drug_hypergeom_fdr,drug_empirical_p_value,drug_empirical_fdr,PMID,NUMBER_OF_ARTICLES,LITERATURE_GENE_TARGETS,Target_Description
0,Buparlisib,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,4,25.0,inhibitor,1-phosphatidylinositol-3-kinase activity,0.000517,0.033046,0.00059,0.038789,"11959846, 15700036, 16807070, 17990317, 166763...",17,PIK3CA,Direct
1,CH-5132799,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,4,25.0,inhibitor,1-phosphatidylinositol-3-kinase activity,0.000517,0.033046,0.00059,0.038789,"11959846, 15700036, 16807070, 17990317, 166763...",17,PIK3CA,Direct
3,Copanlisib,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,4,25.0,inhibitor,1-phosphatidylinositol-3-kinase activity,0.000517,0.033046,0.00059,0.038789,"11959846, 15700036, 16807070, 17990317, 166763...",17,PIK3CA,Direct
5,Golotimod,TLR7,DELETION,1,5,20.0,None,double-stranded RNA binding,0.000016,0.006295,0.00004,0.015664,17201162,1,TLR7,Direct
9,Wortmannin,PIK3CA,SOMATIC,1,5,20.0,None,1-phosphatidylinositol-3-kinase activity,0.000046,0.005476,0.00005,0.006623,"11959846, 15700036, 16807070, 17990317, 166763...",17,PIK3CA,Direct
10,XL765,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,5,20.0,None,1-phosphatidylinositol-3-kinase activity,0.000078,0.008239,0.00012,0.012568,"11959846, 15700036, 16807070, 17990317, 166763...",17,PIK3CA,Direct


In [76]:
hpv_positive_final_indirect

,DRUG,GENE_TARGET,CONNECTED_TO (risk gene),Number of risk or immediate neighbor genes targeted,total_genes_targeted_in_drugbank,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_fdr,drug_empirical_fdr,MUT_TYPE,PMID,NUMBER_OF_ARTICLES,LITERATURE_GENE_TARGETS,RISK_GENE_PMID,RISK_GENE_NUMBER_OF_ARTICLES,RISK_GENE_LITERATURE_GENE_TARGETS,Target_Description
0,1-chloro-6-(4-hydroxyphenyl)-2-naphthol,"ESR1, NCOA1, ESR2, ESR1","PIK3CA, EP300",3,3,100.000000,"inhibitor, inhibitor","14-3-3 protein binding, chromatin binding, DNA...",0.012346,0.014723,"SOMATIC, SOMATIC",11309301,2,ESR1,"11959846, 15700036, 16807070, 17990317, 166763...",17,PIK3CA,Indirect
2,"2-tert-butyl-9-fluoro-1,6-dihydrobenzo[h]imida...","JAK2, TYK2, JAK3, JAK1",PIK3CA,4,5,80.000000,inhibitor,"acetylcholine receptor binding, ATP binding",0.005476,0.008520,SOMATIC,"18204781, 15947106",2,"JAK2, JAK3","11959846, 15700036, 16807070, 17990317, 166763...",17,PIK3CA,Indirect
5,"9,9,9-trifluoro-8-oxo-n-phenylnonanamide","HDAC6, HDAC10, HDAC4, HDAC1, HDAC6, HDAC2, HDAC8","CYLD, EP300",6,7,85.714286,"inhibitor, inhibitor","actin binding, acetylputrescine deacetylase ac...",0.000066,0.002309,"SOMATIC, SOMATIC","12239610, 16773191",3,"HDAC6, HDAC1","16900776, 18497946",2,CYLD,Indirect
6,abrocitinib,"JAK2, TYK2, JAK3, JAK1, JAK2, TYK2, JAK3, JAK1","PIK3CA, PIK3CA",4,4,100.000000,"inhibitor, inhibitor","acetylcholine receptor binding, ATP binding, a...",0.033046,0.038601,"AMPLIFICATION, SOMATIC","18204781, 15947106",4,"JAK2, JAK3","11959846, 15700036, 16807070, 17990317, 166763...",34,PIK3CA,Indirect
8,acetylsalicylic acid,"MYC, TP53, CCND1, MAPK1, TP53, IKBKB, NFKBIA, ...","PIK3CA, CYLD, EP300",7,19,36.842105,"downregulator, inducer, inducer, inhibitor, in...","core promoter sequence-specific DNA binding, 1...",0.005476,0.011401,"SOMATIC, SOMATIC, SOMATIC","18405350, 18075740, 17258495, 15846092, 164905...",200,"PCNA, TP53, NFKBIA, MYC, CCND1","17990317, 15543611, 16676365, 14581353, 169007...",19,"PIK3CA, CYLD",Indirect
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,ulipristal,"PGR, NR3C1, PGR, AR","PIK3CA, EP300",3,3,100.000000,"modulator, antagonist, modulator","ATPase binding, core promoter sequence-specifi...",0.012346,0.014723,"SOMATIC, SOMATIC","18528886, 16835481, 11556855, 17017191, 166904...",12,"PGR, AR","11959846, 15700036, 16807070, 17990317, 166763...",17,PIK3CA,Indirect
198,vatalanib,"EGFR,KDR, EGFR, FLT1, FLT4,EGFR,EGFR,FLT1, EGF...","SOX2,PIK3CA,IGF2BP2,SH3KBP1,VEGFD, PIK3CA",4,4,100.000000,"inhibitor, inhibitor","actin filament binding, ATP binding, actin fil...",0.033046,0.038601,"DELETION,AMPLIFICATION, SOMATIC","15833860, 17258495, 15717992, 17121916, 129520...",1776,EGFR,"11959846, 15700036, 16807070, 17990317, 166763...",35,"PIK3CA, SOX2",Indirect
199,vorinostat,"HDAC6,HDAC6,HDAC8, HDAC2, HDAC3, HDAC1, HDAC1,...","ATXN3L,OFD1,RBBP7,SCML2,TBL1X, CYLD, EP300",5,6,83.333333,"inhibitor, inhibitor, inhibitor","actin binding, actin binding, core promoter se...",0.027227,0.030023,"DELETION, SOMATIC, SOMATIC","12239610, 16773191",10,"HDAC6, HDAC1","16900776, 18497946",2,CYLD,Indirect
203,zanubrutinib,"EGFR,JAK2, FGR, BTK, ERBB4, ITK, LCK, EGFR, JA...","SOX2,PIK3CA,IGF2BP2, PIK3CA",10,15,66.666667,"inhibitor, inhibitor","actin filament binding, acetylcholine receptor...",0.001911,0.003833,"AMPLIFICATION, SOMATIC","17208308, 12649172, 11303632, 14581353, 177629...",1246,"ERBB4, ERBB2, JAK2, EGFR, JAK3","11959846, 15700036, 16807070, 17990317, 166763...",35,"PIK3CA, SOX2",Indirect


## Double check

In [77]:
len(Drug_bank[Drug_bank['drug'].str.lower() == 'artenimol']['gene'].unique())
Drug_bank[Drug_bank['drug'].str.lower() == 'artenimol']['gene'].nunique()

104

In [78]:
Drug_bank[(Drug_bank['drug'].str.lower() == 'quercetin') & (Drug_bank['gene'].str.lower() == 'stat3')]

,drug,polypeptide,gene,gene_description,action,specific_function
11100,Quercetin,Signal transducer and activator of transcripti...,STAT3,Signal transducer and activator of transcripti...,inhibitor,chromatin DNA binding


In [79]:
for gene in hpv_negative_final_indirect[hpv_negative_final_indirect['DRUG'].str.lower() == 'fostamatinib'.lower()]['LITERATURE_GENE_TARGETS']:
    print(gene)

PLK3, RET, AURKA, FGFR3, PRKCI, LYN, SYK, JAK2, MET, EPHB4, EPHB1, MTOR, JAK3, ALK, EPHA1, CDK4, CHEK1, YES1, SRC, ERBB2, PAK1, EPHA2, CDK1, PDGFRA, PLK1, HCK, KIT, FYN, ERBB4, BRAF, LATS1, FGFR2, NTRK3, DAPK1, NTRK1, TTK, DDR2, WEE1, MAP2K3, FGFR1, EGFR, TGFBR1


In [80]:
Drug_bank[(Drug_bank['drug'].str.lower() == 'quercetin'.lower()) & 
          (Drug_bank['gene'].str.lower() == 'stat3'.lower())]

,drug,polypeptide,gene,gene_description,action,specific_function
11100,Quercetin,Signal transducer and activator of transcripti...,STAT3,Signal transducer and activator of transcripti...,inhibitor,chromatin DNA binding


In [81]:
protein_interaction[(protein_interaction['combined_score']>700)&
                    (protein_interaction['Translated_protein_1'].str.lower() == 'stat3') & 
                    (protein_interaction['Translated_protein_2'].str.lower() == 'sox2')]

,protein1,protein2,combined_score,Translated_protein_1,Translated_protein_2
2351081,9606.ENSP00000264657,9606.ENSP00000323588,903,STAT3,SOX2
